In [1]:
# PASO 1 — Base (imports, carpetas, utilidades)
import os, re, csv, json, unicodedata, random
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

# 1) carpetas para datos (CSV) y estado (JSON)
os.makedirs("data", exist_ok=True)
os.makedirs("state", exist_ok=True)

# 2) normalizador simple (minúsculas, sin acentos, quita repeticiones y emojis comunes)
def normalize(text: str) -> str:
    t = text.strip().lower()
    t = ''.join(c for c in unicodedata.normalize('NFKD', t) if not unicodedata.combining(c))
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)          # "siiiii" -> "sii"
    t = re.sub(r'[\u263a-\U0001f645]+', ' ', t)   # emojis comunes
    t = re.sub(r'\s+', ' ', t)
    return t

# 3) lectura/escritura JSON segura (tmp + replace)
def write_json(path: str, data: Any):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def read_json(path: str, default: Any):
    if not os.path.exists(path):
        write_json(path, default)
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# 4) número de pedido tipo Amazon 123-1234567-1234567
def gen_order_number() -> str:
    a = f"{random.randint(100,999)}"
    b = f"{random.randint(0,9999999):07d}"
    c = f"{random.randint(0,9999999):07d}"
    return f"{a}-{b}-{c}"

print("✅ PASO 1 listo — carpetas creadas:", [p for p in os.listdir() if p in ("data","state")])

✅ PASO 1 listo — carpetas creadas: ['data', 'state']


In [2]:
# PASO 2 — Datos iniciales y funciones de carga

import csv

# Rutas
products_path      = "data/products.csv"
variants_path      = "data/product_variants.csv"
orders_path        = "data/orders.csv"
order_items_path   = "data/order_items.csv"
returns_path       = "data/returns.csv"
refunds_path       = "data/refunds.csv"
user_path          = "state/user.json"
cart_path          = "state/cart.json"

# 1) products.csv — product_id,name,category,price_cents,has_sizes,stock
if not os.path.exists(products_path):
    with open(products_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["product_id","name","category","price_cents","has_sizes","stock"])
        w.writerow([1,"Playera básica unisex","ropa",19900,1,""])        # con tallas
        w.writerow([2,"Sudadera con gorro","ropa",59900,1,""])           # con tallas
        w.writerow([3,"Cafetera goteo 12 tazas","hogar",149900,0,15])    # sin tallas (usa 'stock')

# 2) product_variants.csv — variant_id,product_id,size,stock (solo aplica si has_sizes=1)
if not os.path.exists(variants_path):
    with open(variants_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["variant_id","product_id","size","stock"])
        # Playera (S, M, L)
        w.writerow([101,1,"s",12]); w.writerow([102,1,"m",8]); w.writerow([103,1,"l",5])
        # Sudadera (M, L, XL)
        w.writerow([201,2,"m",6]); w.writerow([202,2,"l",4]); w.writerow([203,2,"xl",3])

# 3) orders.csv — pedidos (se llenará al pagar)
if not os.path.exists(orders_path):
    with open(orders_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["order_number","status","subtotal_cents","shipping_cents","total_cents","created_at"])

# 4) order_items.csv — ítems por pedido
if not os.path.exists(order_items_path):
    with open(order_items_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["order_number","product_id","variant_id","size","qty","unit_price_cents"])

# 5) returns.csv — devoluciones
if not os.path.exists(returns_path):
    with open(returns_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["return_id","order_number","item_idx","product_id","variant_id","size","qty","reason","method","status","created_at"])

# 6) refunds.csv — reembolsos
if not os.path.exists(refunds_path):
    with open(refunds_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["refund_id","order_number","amount_cents","status","processed_at"])

# 7) Estado del usuario único (dirección/pago demo) y carrito vacío
_ = read_json(user_path, {
    "user_id": 1,
    "addresses": [{
        "id": 1, "name": "Usuario Demo", "phone": "5512345678",
        "street": "Calle Falsa 123", "city": "CDMX", "state": "CDMX", "postal_code": "06000",
        "is_default": True
    }],
    "payment_methods": [{
        "id": 1, "brand": "visa", "last4": "4242", "token": "tok_demo", "is_default": True
    }]
})
_ = read_json(cart_path, {"items": []})

# 8) Cargadores a memoria
def load_products() -> List[Dict[str, Any]]:
    out = []
    with open(products_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            row["product_id"]  = int(row["product_id"])
            row["price_cents"] = int(row["price_cents"])
            row["has_sizes"]   = int(row["has_sizes"])
            row["stock"]       = int(row["stock"]) if row["stock"] != "" else None
            out.append(row)
    return out

def load_variants() -> Dict[int, List[Dict[str, Any]]]:
    var: Dict[int, List[Dict[str, Any]]] = {}
    with open(variants_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            row["variant_id"] = int(row["variant_id"])
            row["product_id"] = int(row["product_id"])
            row["stock"]      = int(row["stock"])
            var.setdefault(row["product_id"], []).append(row)
    return var

# 9) Cargar ahora y dejar globales
PRODUCTS = load_products()
VARIANTS = load_variants()

print(f"✅ PASO 2 listo — Productos: {len(PRODUCTS)} | Variantes total: {sum(len(v) for v in VARIANTS.values())}")

✅ PASO 2 listo — Productos: 3 | Variantes total: 6


In [3]:
# PASO 3 — Intents/regex y parsers de cantidad/talla
import re

# Afirmaciones/negaciones con slang MX
AFIRMAR = r"\b(si|sí|claro|ok(?:ey)?|vale|va(?: que va)?|sale|sobres|arre|orale|órale|camara|c[aá]mara|de una|jalo|me late|correcto|confirmo|hecho|dale|simon|sim[oó]n)\b"
NEGAR   = r"\b(no+|nel(?: pastel)?|nop|noup|nones|para nada|negativo)\b"

# Globales básicos
ATRAS   = r"\b(atr[aá]s|regresar|volver|me equivoque|me equivoqué|corregir)\b"
SALIR   = r"\b(salir|terminar|cortar conversaci[oó]n|cancelar(?: (?:todo|proceso))?|ya no|olvidalo|olvídalo|hasta (?:aqui|aquí))\b"
AYUDA   = r"\b(ayuda|no entiendo|que puedo hacer|qué puedo hacer|como funciona|cómo funciona)\b"

# Cambiar de operación (lo usaremos para saltar de compra → devolución, etc.)
SWITCH  = r"\b(ahora|mejor)\s+(quiero|vamos a|cambia(?:r)?)\s+(?:a\s+)?(comprar|buscar|carrito|pagar|devolver|reembolso|pedidos?|direcciones?|pagos?)\b"

# Off-topic y lenguaje ofensivo (filtros globales)
ABUSIVE = r"\b(?:pinche[s]?|p[nv]dejo[s]?|idiota[s]?|est[úu]pid[oa]s?|cabr[oó]n(?:es)?|hdp|ptm|mierda|chinga(?:r| tu madre)|piche)\b"
OFFTOP  = r"\b(clima|pron[oó]stico|hora|reloj|d[oó]lar|tipo de cambio|pol[ií]tica|f[uú]tbol|futbol|chiste|cuento)\b"

# Pedidos / devoluciones
ORDER_ID_RE = r"\b\d{3}-\d{7}-\d{7}\b"

INTENTS = {
    # Globales
    "EXIT":      re.compile(SALIR, re.I),
    "BACK":      re.compile(ATRAS, re.I),
    "HELP":      re.compile(AYUDA, re.I),
    "SWITCH":    re.compile(SWITCH, re.I),
    "ABUSIVE":   re.compile(ABUSIVE, re.I),
    "OFF_TOPIC": re.compile(OFFTOP, re.I),

    # Navegación / compras
    "VIEW_CART": re.compile(r"\b(ver carrito|mi carrito|carrito)\b", re.I),
    "PAY":       re.compile(r"\b(pagar|checkout|ir a caja|comprar ahora)\b", re.I),
    "SEARCH":    re.compile(r"\b(buscar|comprar|necesito|quiero|muestrame|mu[eé]strame|ens[eñ]ame|enséñame)\b.*", re.I),
    "SELECT_N":  re.compile(r"^(?:el|la)?\s*(?P<idx>\d+)\b"),

    # Cantidad (set/sumar/restar)
    "QTY_SET":   re.compile(r"(?:pon|ponle|pone|deja|quiero|va a ser|me llevo)\s+(.*)", re.I),
    "QTY_ADD":   re.compile(r"(?:agrega|anade|añade|suma|sube|subele|súbele|mas|más)\s+(.*)", re.I),
    "QTY_SUB":   re.compile(r"(?:quita|resta|baja|bajale|bájale|menos)\s+(.*)", re.I),
    "CONFIRM":   re.compile(AFIRMAR, re.I),
    "DENY":      re.compile(NEGAR, re.I),

    # Carrito (editar por índice; los usaremos después)
    "CART_CLEAR":        re.compile(r"\b(vaciar carrito|borra todo|quita todo)\b", re.I),
    "CART_REMOVE_INDEX": re.compile(r"\b(elimina(?:r)?|quitar|borra(?:r)?)\b.?\b(?:el|la)?\s(?P<idx>\d+)\b", re.I),
    "CART_SET_QTY_IDX":  re.compile(r"\b(?:cambia(?:r)? cantidad|pon|pone|ponle|deja|dejar)\b\s+(?P<qty_text>.+?)\s+(?:al|a la|del|de la)\s+(?P<idx>\d+)\b", re.I),
    "CART_ADD_QTY_IDX":  re.compile(r"\b(?:agrega|anade|añade|suma|sube|subele|súbele|mas|más)\b\s+(?P<qty_text>.+?)\s+(?:al|a la)\s+(?P<idx>\d+)\b", re.I),
    "CART_SUB_QTY_IDX":  re.compile(r"\b(?:quita|resta|baja|bajale|bájale|menos)\b\s+(?P<qty_text>.+?)\s+(?:al|a la)\s+(?P<idx>\d+)\b", re.I),

    # Devolución & reembolso (intenciones de entrada; el flujo lo haremos después)
    "DEV_START":          re.compile(r"\b(devolver|devoluci[oó]n|cambio|no me sirve|no me qued[oa])\b", re.I),
    "REFUND_STATUS":      re.compile(r"\b(reembolso|estatus del reembolso|cuando me (?:depositan|devuelven))\b", re.I),
    "ORDER_ID_OR_RECENT": re.compile(rf"({ORDER_ID_RE}|el mas reciente|el más reciente|ultimo|último|mas reciente|más reciente)", re.I),

    # Motivo y método (para el flujo de devolución más adelante)
    "DEV_REASON":         re.compile(r"\b(no es lo que ped[ií]|dañad[oa]|golpead[oa]|incompleto|defectuoso|no funciona|me equivoqu[eé]|tarde|otro|otra cosa)\b", re.I),
    "DEV_METHOD":         re.compile(r"\b(etiqueta|paqueter[ií]a|punto de entrega|drop ?off|tienda|sucursal|recolecci[oó]n)\b", re.I),

    # Admin (direcciones y pagos) — los implementamos después
    "ADMIN_DIR":          re.compile(r"\b(direcciones?|domicilios?)\b", re.I),
    "ADMIN_PAGO":         re.compile(r"\b(pagos?|tarjetas?|metodo de pago|m[eé]todo de pago)\b", re.I),
}

# ---- Parseo de CANTIDAD ----
WORDS = {
    "cero":0,"un":1,"una":1,"uno":1,"dos":2,"tres":3,"cuatro":4,"cinco":5,"seis":6,
    "siete":7,"ocho":8,"nueve":9,"diez":10,"once":11,"doce":12
}
ESPECIALES = {"par":2, "media docena":6, "docena":12}
RE_NUM = re.compile(r"(?P<num>\d+)")
RE_ESP = re.compile(r"(media docena|docena|par)")
RE_PAL = re.compile("|".join(sorted(map(re.escape, WORDS.keys()), key=len, reverse=True)))

def parse_qty(fragment: str) -> Optional[int]:
    frag = normalize(fragment)
    m = RE_NUM.search(frag)
    if m: return int(m.group("num"))
    m = RE_ESP.search(frag)
    if m: return ESPECIALES[m.group(1)]
    m = RE_PAL.search(frag)
    if m: return WORDS[m.group(0)]
    return None

# ---- Parseo de TALLA ----
TALLA_EQ = {"extra chica":"xs","chica":"s","ch":"s","mediana":"m","med":"m","grande":"l","extra grande":"xl"}
RE_TALLA = re.compile(r"(?:talla\s*)?(?P<talla>xs|s|m|l|xl|xxl|extra chica|chica|ch|mediana|med|grande|extra grande)\b")

def parse_size(text: str) -> Optional[str]:
    t = normalize(text)
    m = RE_TALLA.search(t)
    if not m: return None
    raw = m.group("talla")
    return TALLA_EQ.get(raw, raw)

# Pruebas rápidas (deberías ver 12, 3, s, xl)
print("Qty 'docena' →", parse_qty("docena"))
print("Qty 'pon 3'  →", parse_qty("pon 3"))
print("Talla 'ch'   →", parse_size("ch"))
print("Talla 'talla XL' →", parse_size("talla XL"))
print("✅ PASO 3 listo")

Qty 'docena' → 12
Qty 'pon 3'  → 3
Talla 'ch'   → s
Talla 'talla XL' → xl
✅ PASO 3 listo


In [4]:
# PASO 4 — Funciones de negocio: productos, stock, carrito y checkout
from datetime import datetime
import csv

# ---------- utilidades de carga/estado ----------
def load_cart() -> Dict[str, Any]:
    """Lee el carrito del JSON (si no existe, uno vacío)."""
    return read_json(cart_path, {"items": []})

def save_cart(cart: Dict[str, Any]):
    """Guarda el carrito en disco de forma atómica."""
    write_json(cart_path, cart)

def load_products_if_changed():
    """Recarga CSV de productos y variantes (por cambios de stock tras pagar)."""
    global PRODUCTS, VARIANTS
    PRODUCTS = load_products()
    VARIANTS = load_variants()

# ---------- consulta de productos ----------
def product_price_cents(pid: int) -> int:
    for p in PRODUCTS:
        if p["product_id"] == pid:
            return p["price_cents"]
    raise KeyError(f"Producto {pid} no existe")

def list_variants(product_id: int) -> List[Dict[str, Any]]:
    """Variantes (tallas) de un producto de ropa."""
    return VARIANTS.get(product_id, [])

def search_products(query: str) -> List[Dict[str, Any]]:
    """
    Busca por nombre o categoría (búsqueda muy simple).
    Si no hay match, regresa todos para que el usuario elija.
    """
    q = normalize(query)
    results = []
    for p in PRODUCTS:
        if (q in normalize(p["name"])) or (q in normalize(p["category"])):
            results.append(p)
    return results or PRODUCTS

def find_product_by_index(results: List[Dict[str, Any]], idx: int) -> Optional[Dict[str, Any]]:
    """Del listado mostrado (1..N) toma el producto por número."""
    if 1 <= idx <= len(results):
        return results[idx-1]
    return None

# ---------- stock ----------
def stock_available(pid: int, size: Optional[str], qty: int) -> bool:
    """
    Valida si hay stock suficiente:
      - Si es ropa (has_sizes=1): stock por talla (size).
      - Si NO es ropa: stock del producto.
    """
    p = next(p for p in PRODUCTS if p["product_id"] == pid)
    if p["has_sizes"]:
        vs = [v for v in list_variants(pid) if v["size"] == size]
        if not vs:
            return False
        return vs[0]["stock"] >= qty
    else:
        return (p["stock"] or 0) >= qty

def adjust_stock_on_checkout(items: List[Dict[str, Any]]):
    """
    Descuenta el stock en los CSV al confirmar el pedido.
    - Para ropa: descuenta en product_variants.csv (por variant_id).
    - Para productos sin talla: descuenta en products.csv.
    """
    # 1) Variantes (ropa)
    with open(variants_path, encoding="utf-8") as f:
        var_rows = list(csv.DictReader(f))
    for it in items:
        if it["variant_id"]:
            for row in var_rows:
                if int(row["variant_id"]) == it["variant_id"]:
                    row["stock"] = str(int(row["stock"]) - it["qty"])
    with open(variants_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["variant_id","product_id","size","stock"])
        w.writeheader()
        for r in var_rows:
            w.writerow(r)

    # 2) Productos sin talla
    with open(products_path, encoding="utf-8") as f:
        prod_rows = list(csv.DictReader(f))
    for it in items:
        if not it["variant_id"]:
            for row in prod_rows:
                if int(row["product_id"]) == it["product_id"]:
                    row["stock"] = str(int(row["stock"]) - it["qty"])
    with open(products_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["product_id","name","category","price_cents","has_sizes","stock"])
        w.writeheader()
        for r in prod_rows:
            w.writerow(r)

    load_products_if_changed()

# ---------- carrito: agregar y ver ----------
def cart_add(pid: int, qty: int, size: Optional[str]) -> tuple[bool,str]:
    """
    Añade al carrito (merge si ya existe mismo producto+talla).
    Valida talla requerida y stock.
    """
    p = next(p for p in PRODUCTS if p["product_id"] == pid)
    variant_id = None

    if p["has_sizes"]:
        if not size:
            return False, "Falta la talla."
        vs = [v for v in list_variants(pid) if v["size"] == size]
        if not vs:
            return False, "Esa talla no existe para este producto."
        if vs[0]["stock"] < qty:
            return False, f"No hay stock suficiente de talla {size}."
        variant_id = vs[0]["variant_id"]
    else:
        if (p["stock"] or 0) < qty:
            return False, "No hay stock suficiente."

    cart = load_cart()

    # Si ya existe el mismo producto+talla, solo incrementa cantidad
    for it in cart["items"]:
        if it["product_id"] == pid and it.get("size") == size:
            new_qty = it["qty"] + qty
            if not stock_available(pid, size, new_qty):
                return False, "No hay stock suficiente para sumar esa cantidad."
            it["qty"] = new_qty
            save_cart(cart)
            return True, "Cantidad actualizada en el carrito."

    # Nuevo renglón en el carrito
    cart["items"].append({
        "product_id": pid,
        "variant_id": variant_id,  # None para sin talla
        "size": size,              # None para sin talla
        "qty": qty,
        "unit_price_cents": product_price_cents(pid)
    })
    save_cart(cart)
    return True, "Agregado al carrito."

def cart_summary() -> str:
    """Regresa un texto legible del carrito con total estimado."""
    cart = load_cart()
    if not cart["items"]:
        return "Tu carrito está vacío."
    lines, total = [], 0
    for i, it in enumerate(cart["items"], 1):
        p = next(p for p in PRODUCTS if p["product_id"] == it["product_id"])
        desc = f"{i}) {p['name']}"
        if it["size"]:
            desc += f" talla {it['size'].upper()}"
        desc += f" — {it['qty']} pza(s) — ${it['unit_price_cents']/100:.2f} c/u"
        lines.append(desc)
        total += it["unit_price_cents"] * it["qty"]
    lines.append(f"Total estimado: ${total/100:.2f}")
    return "\n".join(lines)

# ---------- carrito: editar por índice ----------
def cart_remove_index(idx: int) -> str:
    cart = load_cart()
    if not (1 <= idx <= len(cart["items"])):
        return "Índice inválido."
    cart["items"].pop(idx-1)
    save_cart(cart)
    return "Artículo eliminado del carrito."

def cart_set_qty_index(idx: int, qty: int) -> str:
    cart = load_cart()
    if not (1 <= idx <= len(cart["items"])):
        return "Índice inválido."
    if qty < 1:
        cart["items"].pop(idx-1)
        save_cart(cart)
        return "Cantidad 0: artículo eliminado."
    it = cart["items"][idx-1]
    if not stock_available(it["product_id"], it.get("size"), qty):
        return "No hay stock suficiente para esa cantidad."
    it["qty"] = qty
    save_cart(cart)
    return "Cantidad actualizada."

def cart_add_qty_index(idx: int, delta: int) -> str:
    cart = load_cart()
    if not (1 <= idx <= len(cart["items"])):
        return "Índice inválido."
    it = cart["items"][idx-1]
    new_qty = it["qty"] + max(1, delta)
    if not stock_available(it["product_id"], it.get("size"), new_qty):
        return "No hay stock suficiente para sumar esa cantidad."
    it["qty"] = new_qty
    save_cart(cart)
    return "Cantidad aumentada."

def cart_sub_qty_index(idx: int, delta: int) -> str:
    cart = load_cart()
    if not (1 <= idx <= len(cart["items"])):
        return "Índice inválido."
    it = cart["items"][idx-1]
    new_qty = it["qty"] - max(1, delta)
    if new_qty <= 0:
        cart["items"].pop(idx-1)
        save_cart(cart)
        return "Cantidad llegó a 0: artículo eliminado."
    if not stock_available(it["product_id"], it.get("size"), new_qty):
        return "Cantidad inválida."
    it["qty"] = new_qty
    save_cart(cart)
    return "Cantidad reducida."

def cart_clear() -> str:
    save_cart({"items": []})
    return "Carrito vaciado."

# ---------- checkout (crear pedido, descontar stock y vaciar carrito) ----------
def checkout() -> str:
    cart = load_cart()
    if not cart["items"]:
        return "No tienes artículos en el carrito."

    # Validación final de stock (por si cambió)
    for it in cart["items"]:
        if not stock_available(it["product_id"], it.get("size"), it["qty"]):
            return "Ya no hay stock para algún item. Ajusta cantidades."

    subtotal = sum(it["unit_price_cents"] * it["qty"] for it in cart["items"])
    shipping = 0
    total = subtotal + shipping
    order_number = gen_order_number()

    # Guardar pedido
    with open(orders_path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([order_number, "paid", subtotal, shipping, total, datetime.now().isoformat(timespec="seconds")])

    # Guardar items del pedido
    with open(order_items_path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for it in cart["items"]:
            w.writerow([order_number, it["product_id"], it.get("variant_id") or "", it.get("size") or "", it["qty"], it["unit_price_cents"]])

    # Descontar stock y vaciar carrito
    adjust_stock_on_checkout(cart["items"])
    save_cart({"items": []})

    return f"¡Pedido confirmado! Número: {order_number}"

# ---------- lectores de pedidos e items (los usaremos para devoluciones/reembolsos) ----------
def list_orders() -> List[Dict[str, Any]]:
    orders = []
    with open(orders_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            row["subtotal_cents"] = int(row["subtotal_cents"])
            row["shipping_cents"] = int(row["shipping_cents"])
            row["total_cents"]    = int(row["total_cents"])
            orders.append(row)
    # más recientes primero
    orders.sort(key=lambda x: x["created_at"], reverse=True)
    return orders

def list_order_items(order_number: str) -> List[Dict[str, Any]]:
    items = []
    with open(order_items_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            if row["order_number"] == order_number:
                row["product_id"]       = int(row["product_id"])
                row["variant_id"]       = int(row["variant_id"]) if row["variant_id"] else None
                row["qty"]              = int(row["qty"])
                row["unit_price_cents"] = int(row["unit_price_cents"])
                # nombre del producto
                p = next(p for p in PRODUCTS if p["product_id"] == row["product_id"])
                row["product_name"]     = p["name"]
                items.append(row)
    return items

print("✅ PASO 4 listo — funciones de negocio cargadas")

✅ PASO 4 listo — funciones de negocio cargadas


In [5]:
# PASO 5 — Autómata conversacional base (compras, carrito, checkout) + filtros globales
from dataclasses import dataclass, field

@dataclass
class Ctx:
    state: str = "INICIO"
    last_results: List[Dict[str, Any]] = field(default_factory=list)
    current_product: Optional[Dict[str, Any]] = None
    qty: Optional[int] = None
    size: Optional[str] = None

class AmazonBot:
    def __init__(self):
        self.ctx = Ctx()
        self.abusive_count = 0
        self.offtopic_count = 0

    # --- Mensaje de ayuda (dinámico) ---
    def help_text(self):
        return ("Puedo: *buscar/comprar, **ver/editar carrito* y *pagar*.\n"
                "Comandos: ‘ver carrito’, ‘pagar’, ‘atrás’, ‘salir’.\n"
                "También entiendo: ‘ahora quiero [carrito|pagar|comprar]’. "
                "Los módulos de *devolución, reembolso, pedidos, direcciones y pagos* "
                "los activaremos en los siguientes pasos.")

    # --- Prompt por estado ---
    def prompt(self) -> str:
        s = self.ctx.state
        if s == "INICIO":
            return ("¿Qué estás buscando? (ej. 'playera', 'sudadera', 'cafetera')\n"
                    "También: 'ver carrito', 'pagar', 'ayuda', 'salir'.")
        if s == "LISTA":
            return "Elige un producto por número (ej. '1'), o escribe otra búsqueda."
        if s == "DETALLE":
            p = self.ctx.current_product
            base = f"Seleccionado: {p['name']} — ${p['price_cents']/100:.2f}\n"
            if int(p["has_sizes"]):
                base += "Este producto requiere talla (XS,S,M,L,XL,XXL). "
            base += "Indica cantidad (ej. 'pon 2', 'par', 'docena') y, si aplica, la talla (ej. 'talla M')."
            return base
        if s == "CONFIRMAR":
            parts = [f"cantidad {self.ctx.qty}"]
            if self.ctx.size:
                parts.append(f"talla {self.ctx.size.upper()}")
            return f"¿Confirmas {', '.join(parts)}? (sí/no)"
        if s == "CARRITO":
            ayuda = ("\n\nComandos: 'pagar', 'vaciar carrito', "
                     "'elimina 2', 'cambia cantidad del 1 a 3', "
                     "'pon 2 al 1', 'suma 1 al 2', 'quita 1 al 2', 'seguir buscando'.")
            return cart_summary() + ayuda
        if s == "FIN":
            return "¡Gracias! Hasta luego."
        return ""

    # --- Router principal ---
    def handle(self, user: str) -> str:
        u = normalize(user)

        # 0) Filtros globales: ofensivo / off-topic
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"
                return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Seguimos con tu compra o necesitas ver el carrito?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("Eso es otro tema. " + self.help_text())
            return ("Creo que te saliste del tema 😅. " + self.help_text())
        self.offtopic_count = 0

        # 1) Comandos globales
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"
            return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            if self.ctx.state in ("DETALLE","CONFIRMAR"):
                self.ctx.state = "LISTA"
                return self.prompt()
            self.ctx.state = "INICIO"
            return self.prompt()
        if INTENTS["HELP"].search(u):
            return self.help_text()
        if INTENTS["VIEW_CART"].search(u):
            self.ctx.state = "CARRITO"
            return self.prompt()
        if INTENTS["PAY"].search(u):
            msg = checkout()
            # tras pagar, volvemos a INICIO
            self.ctx.state = "INICIO"
            return msg + "\n\n" + self.prompt()

        # 1.b Cambiar de operación (SWITCH)
        m_switch = INTENTS["SWITCH"].search(u)
        if m_switch:
            target = normalize(m_switch.group(3))  # grupo con la palabra clave destino
            if target in ("comprar","buscar"):
                self.ctx.state = "INICIO"
                return "Cambiamos a compras. " + self.prompt()
            if target == "carrito":
                self.ctx.state = "CARRITO"
                return "Listo, aquí está tu carrito:\n\n" + self.prompt()
            if target == "pagar":
                msg = checkout()
                self.ctx.state = "INICIO"
                return "Cambiado a pagar.\n" + msg + "\n\n" + self.prompt()
            # Lo demás lo habilitaremos más adelante:
            if target in ("devolver","reembolso","pedidos","pedido","direcciones","pagos","pago"):
                return ("¡Anotado! Ese módulo lo activamos en los siguientes pasos. "
                        "Por ahora puedo comprar, ver carrito y pagar. ¿Qué deseas hacer?")
            # fallback
            return "Cambio entendido, pero no reconocí la operación. " + self.help_text()

        # 2) Flujo por estado
        s = self.ctx.state

        # INICIO → LISTA (cualquier texto lo tratamos como búsqueda)
        if s == "INICIO":
            self.ctx.last_results = search_products(u)
            if not self.ctx.last_results:
                return "No encontré resultados. Intenta con otra palabra."
            self.ctx.state = "LISTA"
            lines = ["Resultados:"]
            for i,p in enumerate(self.ctx.last_results, 1):
                tag = " (tallas)" if int(p["has_sizes"]) else ""
                lines.append(f"{i}) {p['name']} — ${p['price_cents']/100:.2f}{tag}")
            return "\n".join(lines) + "\n\n" + self.prompt()

        # LISTA: elegir producto por número o re-buscar
        if s == "LISTA":
            m = INTENTS["SELECT_N"].search(u)
            if m:
                idx = int(m.group("idx"))
                prod = find_product_by_index(self.ctx.last_results, idx)
                if not prod:
                    return "Índice inválido. Elige un número de la lista."
                self.ctx.current_product = prod
                self.ctx.qty = None
                self.ctx.size = None
                self.ctx.state = "DETALLE"
                return self.prompt()
            # Si escribe otra búsqueda, reusamos INICIO
            self.ctx.state = "INICIO"
            return self.handle(u)

        # DETALLE: capturar cantidad y (si aplica) talla
        if s == "DETALLE":
            p = self.ctx.current_product

            # cantidad (set/add/sub) o “solo número”
            if INTENTS["QTY_ADD"].search(u):
                frag = INTENTS["QTY_ADD"].search(u).group(1)
                num = parse_qty(frag)
                if num is None: return "No entendí cuánto sumar."
                self.ctx.qty = (self.ctx.qty or 0) + num
            elif INTENTS["QTY_SUB"].search(u):
                frag = INTENTS["QTY_SUB"].search(u).group(1)
                num = parse_qty(frag)
                if num is None: return "No entendí cuánto quitar."
                self.ctx.qty = max(1, (self.ctx.qty or 1) - num)
            elif INTENTS["QTY_SET"].search(u):
                frag = INTENTS["QTY_SET"].search(u).group(1)
                num = parse_qty(frag)
                if num is None: return "No entendí la cantidad."
                self.ctx.qty = max(1, num)
            else:
                num = parse_qty(u)
                if num: self.ctx.qty = max(1, num)

            # talla (si aplica)
            sz = parse_size(u)
            if sz: self.ctx.size = sz

            # validaciones
            if int(p["has_sizes"]) and not self.ctx.size:
                return "Este producto requiere talla (XS,S,M,L,XL,XXL). Dime la talla (ej. 'talla M')."
            if not self.ctx.qty:
                return "¿Cuántas piezas quieres? (ej. 'pon 2', 'par', 'docena')"
            if not stock_available(p["product_id"], self.ctx.size, self.ctx.qty):
                return "No hay stock suficiente. Prueba con menos cantidad o otra talla."

            self.ctx.state = "CONFIRMAR"
            return self.prompt()

        # CONFIRMAR: agrega al carrito o corrige
        if s == "CONFIRMAR":
            if INTENTS["CONFIRM"].search(u):
                ok, msg = cart_add(self.ctx.current_product["product_id"], self.ctx.qty, self.ctx.size)
                self.ctx.state = "CARRITO" if ok else "DETALLE"
                return msg + ("\n\n" + self.prompt() if ok else "")
            if INTENTS["DENY"].search(u):
                self.ctx.state = "DETALLE"
                return "Sin problema, dime otra cantidad/talla."
            # si escribe cantidad/talla aquí, re-procesamos en DETALLE
            self.ctx.state = "DETALLE"
            return self.handle(u)

        # CARRITO: edición por índice, vaciar o seguir comprando
        if s == "CARRITO":
            if INTENTS["CART_CLEAR"].search(u):
                msg = cart_clear()
                return msg + "\n\n" + self.prompt()
            m = INTENTS["CART_REMOVE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                msg = cart_remove_index(idx)
                return msg + "\n\n" + self.prompt()
            m = INTENTS["CART_SET_QTY_IDX"].search(u)
            if m:
                idx = int(m.group("idx")); qtxt = m.group("qty_text")
                qty = parse_qty(qtxt)
                if qty is None: return "No entendí la cantidad. Ej: 'cambia cantidad del 1 a 3'."
                msg = cart_set_qty_index(idx, qty)
                return msg + "\n\n" + self.prompt()
            m = INTENTS["CART_ADD_QTY_IDX"].search(u)
            if m:
                idx = int(m.group("idx")); qtxt = m.group("qty_text")
                qty = parse_qty(qtxt)
                if qty is None: return "No entendí cuánto sumar. Ej: 'suma 2 al 1'."
                msg = cart_add_qty_index(idx, qty)
                return msg + "\n\n" + self.prompt()
            m = INTENTS["CART_SUB_QTY_IDX"].search(u)
            if m:
                idx = int(m.group("idx")); qtxt = m.group("qty_text")
                qty = parse_qty(qtxt)
                if qty is None: return "No entendí cuánto quitar. Ej: 'quita 1 al 2'."
                msg = cart_sub_qty_index(idx, qty)
                return msg + "\n\n" + self.prompt()
            # seguir comprando (escribe otra búsqueda aquí mismo)
            if INTENTS["SEARCH"].search(u):
                self.ctx.state = "INICIO"
                return self.handle(u)
            return self.prompt()

        # Fallback: volver a INICIO
        self.ctx.state = "INICIO"
        return self.prompt()

print("✅ PASO 5 listo — Bot base definido (compras/carrito/checkout + globales)")

✅ PASO 5 listo — Bot base definido (compras/carrito/checkout + globales)


In [6]:
# PASO 6 — Devoluciones: helpers + clase AmazonBotV2 (extiende compras con flujo de devolución)
from datetime import datetime, timedelta
from dataclasses import dataclass, field

# --- Helpers para devoluciones ---
def _parse_dt(s: str) -> datetime:
    if not s:
        return datetime.now()
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return datetime.now()

def return_eligible(order_row: Dict[str, Any]) -> bool:
    # Ventana de 30 días desde la creación del pedido
    dt = _parse_dt(order_row.get("created_at", ""))
    return dt >= (datetime.now() - timedelta(days=30))

def _next_return_id() -> int:
    rid = 1
    with open(returns_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            try: rid = max(rid, int(row["return_id"]) + 1)
            except: pass
    return rid

def _next_refund_id() -> int:
    rid = 1
    with open(refunds_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            try: rid = max(rid, int(row["refund_id"]) + 1)
            except: pass
    return rid

def create_return(order_number: str, item_idx: int, qty: int, reason: str, method: str) -> Dict[str, Any]:
    """Crea una devolución y registra un reembolso 'pending' (simple: precio unitario * qty)."""
    items = list_order_items(order_number)
    if not (1 <= item_idx <= len(items)):
        return {"ok": False, "msg": "Índice de artículo inválido."}
    it = items[item_idx-1]
    if qty < 1 or qty > it["qty"]:
        return {"ok": False, "msg": f"Cantidad inválida (1–{it['qty']})."}

    rid = _next_return_id()
    created_at = datetime.now().isoformat(timespec="seconds")

    with open(returns_path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([
            rid, order_number, item_idx, it["product_id"], it["variant_id"] or "", it.get("size") or "",
            qty, reason, method, "requested", created_at
        ])

    # Reembolso estimado (pendiente)
    amount = it["unit_price_cents"] * qty
    ref_id = _next_refund_id()
    with open(refunds_path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([ref_id, order_number, amount, "pending", ""])

    return {"ok": True, "msg": f"Devolución #{rid} creada. Reembolso estimado ${amount/100:.2f} (pendiente)."}

# --- Contexto extendido solo para devolución (mantiene el mismo self.ctx.state) ---
@dataclass
class DevData:
    order: Optional[str] = None
    items: List[Dict[str, Any]] = field(default_factory=list)
    item_idx: Optional[int] = None
    qty: Optional[int] = None
    reason: Optional[str] = None
    method: Optional[str] = None

class AmazonBotV2(AmazonBot):
    def __init__(self):
        super().__init__()      # conserva INICIO, LISTA, DETALLE, CONFIRMAR, CARRITO
        self.dev = DevData()    # datos temporales de devolución

    # mapear texto → método interno
    def _map_method(self, u: str) -> Optional[str]:
        t = normalize(u)
        if re.search(r"etiqueta|paqueter", t): return "etiqueta"
        if re.search(r"punto de entrega|drop ?off", t): return "punto_entrega"
        if re.search(r"tienda|sucursal", t): return "tienda"
        if re.search(r"recolecci[oó]n", t): return "recoleccion"
        return None

    # prompts para los estados nuevos
    def prompt(self) -> str:
        s = self.ctx.state
        if s == "DEV_INICIO":
            return "¿De qué pedido quieres devolver? Dime el número (123-1234567-1234567) o 'el más reciente'."
        if s == "DEV_LISTA_ITEMS":
            lines = [f"Pedido {self.dev.order}. ¿Qué artículo quieres devolver?"]
            for i,it in enumerate(self.dev.items, 1):
                size = f" talla {it['size'].upper()}" if it.get("size") else ""
                lines.append(f"{i}) {it['product_name']}{size} — {it['qty']} pza(s)")
            lines.append("Responde con el número (ej. '1').")
            return "\n".join(lines)
        if s == "DEV_CANTIDAD":
            it = self.dev.items[self.dev.item_idx-1]
            return f"¿Cuántas unidades vas a devolver? (1–{it['qty']}). Puedes decir 'todo'."
        if s == "DEV_MOTIVO":
            return "¿Cuál es el motivo? (no es lo que pedí/dañado/defectuoso/no funciona/me equivoqué/tarde/otro)"
        if s == "DEV_METODO":
            return "Elige método: etiqueta de paquetería, punto de entrega, tienda o recolección."
        if s == "DEV_CONFIRMAR":
            it = self.dev.items[self.dev.item_idx-1]
            size = f" talla {it['size'].upper()}" if it.get("size") else ""
            return (f"Vas a devolver {self.dev.qty} pza(s) de '{it['product_name']}{size}' "
                    f"del pedido {self.dev.order}. Motivo: {self.dev.reason}. "
                    f"Método: {self.dev.method}. ¿Confirmas? (sí/no)")
        # todo lo demás lo maneja el prompt de compras
        return super().prompt()

    def handle(self, user: str) -> str:
        u = normalize(user)

        # 0) ofensivo / off-topic (misma lógica que el base)
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"
                return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Seguimos con tu compra o tramitamos una devolución?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("Puedo ayudarte a comprar, ver/editar el carrito, pagar y *devolver*. "
                        "Prueba: 'playera', 'ver carrito', 'pagar' o 'devolver'.")
            return ("Creo que eso es otro tema 😅. Puedo buscar productos, mostrar el carrito, pagar o *tramitar devoluciones*.")
        self.offtopic_count = 0

        # 1) globales
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            # Si estamos en flujo DEV, retrocedemos un paso
            if self.ctx.state == "DEV_CONFIRMAR":
                self.ctx.state = "DEV_METODO"; return self.prompt()
            if self.ctx.state == "DEV_METODO":
                self.ctx.state = "DEV_MOTIVO"; return self.prompt()
            if self.ctx.state == "DEV_MOTIVO":
                self.ctx.state = "DEV_CANTIDAD"; return self.prompt()
            if self.ctx.state == "DEV_CANTIDAD":
                self.ctx.state = "DEV_LISTA_ITEMS"; return self.prompt()
            if self.ctx.state == "DEV_LISTA_ITEMS":
                self.ctx.state = "DEV_INICIO"; return self.prompt()
            # fuera de DEV: usa el comportamiento base
            return super().handle(user)
        if INTENTS["VIEW_CART"].search(u) or INTENTS["PAY"].search(u) or INTENTS["SEARCH"].search(u):
            # Cambiar de operación a mitad del flujo DEV → delega al flujo de compras
            return super().handle(user)

        # 2) Entrada al flujo DEV desde cualquier estado
        if INTENTS["DEV_START"].search(u) and not str(self.ctx.state).startswith("DEV_"):
            self.dev = DevData()                # limpia datos anteriores
            self.ctx.state = "DEV_INICIO"
            return self.prompt()

        # 3) Flujo DEV por estados
        s = self.ctx.state

        if s == "DEV_INICIO":
            m = INTENTS["ORDER_ID_OR_RECENT"].search(u)
            if not m:
                ods = list_orders()
                if not ods:
                    self.ctx.state = "INICIO"; return "No tienes pedidos aún. ¿Quieres comprar algo?"
                lines = ["Pedidos (más recientes primero):"]
                for i,o in enumerate(ods,1):
                    lines.append(f"{i}) {o['order_number']} — ${o['total_cents']/100:.2f}")
                lines.append("Dime el número de pedido (123-1234567-1234567) o 'el más reciente'.")
                return "\n".join(lines)

            key = m.group(1)
            if "reciente" in key or "ultimo" in key or "último" in key:
                ods = list_orders()
                if not ods: 
                    self.ctx.state = "INICIO"; return "No hay pedidos registrados."
                chosen = ods[0]
            else:
                found = [o for o in list_orders() if o["order_number"] == key]
                if not found:
                    return "No ubico ese pedido. Verifica el formato 123-1234567-1234567."
                chosen = found[0]

            if not return_eligible(chosen):
                self.ctx.state = "INICIO"
                return "Ese pedido ya no es elegible para devolución (ventana de 30 días). Si quieres, te paso a soporte."

            self.dev.order = chosen["order_number"]
            self.dev.items = list_order_items(self.dev.order)
            if not self.dev.items:
                self.ctx.state = "INICIO"; return "Ese pedido no tiene artículos."
            self.ctx.state = "DEV_LISTA_ITEMS"; return self.prompt()

        if s == "DEV_LISTA_ITEMS":
            m = INTENTS["SELECT_N"].search(u)
            if not m:
                return "Elige un artículo por número (ej. '1')."
            idx = int(m.group("idx"))
            if not (1 <= idx <= len(self.dev.items)):
                return "Índice inválido. Elige un número de la lista."
            self.dev.item_idx = idx
            self.ctx.state = "DEV_CANTIDAD"; return self.prompt()

        if s == "DEV_CANTIDAD":
            it = self.dev.items[self.dev.item_idx-1]
            if "todo" in u or "todas" in u or "toda" in u:
                self.dev.qty = it["qty"]
            else:
                q = parse_qty(u)
                if q is None: return f"No entendí la cantidad. Di un número entre 1 y {it['qty']} o 'todo'."
                if not (1 <= q <= it["qty"]): return f"Cantidad inválida (1–{it['qty']})."
                self.dev.qty = q
            self.ctx.state = "DEV_MOTIVO"; return self.prompt()

        if s == "DEV_MOTIVO":
            m = INTENTS["DEV_REASON"].search(u)
            self.dev.reason = m.group(0) if m else "otro"
            self.ctx.state = "DEV_METODO"; return self.prompt()

        if s == "DEV_METODO":
            m = INTENTS["DEV_METHOD"].search(u)
            method = self._map_method(m.group(0) if m else u)
            if not method:
                return "No entendí el método. Di: etiqueta, punto de entrega, tienda o recolección."
            self.dev.method = method
            self.ctx.state = "DEV_CONFIRMAR"; return self.prompt()

        if s == "DEV_CONFIRMAR":
            if INTENTS["CONFIRM"].search(u):
                r = create_return(self.dev.order, self.dev.item_idx, self.dev.qty, self.dev.reason, self.dev.method)
                self.ctx.state = "INICIO" if r["ok"] else "DEV_LISTA_ITEMS"
                return r["msg"] + ("\n\n" + super().prompt() if r["ok"] else "")
            if INTENTS["DENY"].search(u):
                self.ctx.state = "DEV_METODO"; return "Sin problema, elige otro método."
            # si vuelve a declarar cantidad/motivo/método
            if INTENTS["DEV_METHOD"].search(u):
                self.ctx.state = "DEV_METODO"; return self.handle(u)
            if INTENTS["DEV_REASON"].search(u):
                self.ctx.state = "DEV_MOTIVO"; return self.handle(u)
            q = parse_qty(u)
            if q is not None:
                self.ctx.state = "DEV_CANTIDAD"; return self.handle(u)
            return "¿Confirmas la devolución? (sí/no)"

        # 4) fuera del flujo DEV, usa el flujo de compras normal
        return super().handle(user)

print("✅ PASO 6 listo — Devoluciones integradas en AmazonBotV2")

✅ PASO 6 listo — Devoluciones integradas en AmazonBotV2


In [7]:
# PASO 7 — Seguimiento de reembolso: helper + clase AmazonBotV3

import csv, re
from dataclasses import dataclass

def refund_status(order_number: Optional[str]=None) -> str:
    """
    Devuelve un texto con el estatus de devoluciones y reembolsos del pedido.
    Si order_number es None, toma el pedido más reciente (si existe).
    """
    ods = list_orders()
    if not ods:
        return "No hay pedidos registrados."

    if order_number is None:
        order_number = ods[0]["order_number"]

    refunds = []
    with open(refunds_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            if row["order_number"] == order_number:
                row["amount_cents"] = int(row["amount_cents"])
                refunds.append(row)

    returns = []
    with open(returns_path, encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            if row["order_number"] == order_number:
                returns.append(row)

    if not refunds and not returns:
        return f"No hay devoluciones/reembolsos para el pedido {order_number}."

    lines = [f"Pedido {order_number} — Estatus:"]
    for rr in returns:
        lines.append(f"- Devolución #{rr['return_id']}: {rr['status']} (método: {rr['method']}, creado {rr['created_at']})")
    for rf in refunds:
        est = rf['status'] or 'pendiente'
        lines.append(f"- Reembolso #{rf['refund_id']}: ${rf['amount_cents']/100:.2f} — {est}")
    return "\n".join(lines)

class AmazonBotV3(AmazonBotV2):
    def __init__(self):
        super().__init__()      # conserva INICIO, LISTA, DETALLE, CONFIRMAR, CARRITO

    # añadimos prompt para el estado REFUND_INFO
    def prompt(self) -> str:
        if self.ctx.state == "REFUND_INFO":
            return "¿De qué pedido quieres ver el reembolso? Dime el número (123-1234567-1234567) o 'el más reciente'."
        return super().prompt()

    def handle(self, user: str) -> str:
        u = normalize(user)

        # Filtros globales (mismos que ya tienes)
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Seguimos con tu compra o con tu devolución?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("Puedo ayudarte a comprar, ver/editar el carrito, pagar, *devolver* y ver *reembolso*. "
                        "Ejemplos: 'playera', 'ver carrito', 'pagar', 'devolver', 'reembolso'.")
            return ("Creo que eso es otro tema 😅. Puedo buscar productos, pagar, tramitar devoluciones o revisar reembolsos.")
        self.offtopic_count = 0

        # Globales
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            # si estamos pidiendo info de reembolso, regresar a inicio
            if self.ctx.state == "REFUND_INFO":
                self.ctx.state = "INICIO"; return self.prompt()
            # en caso contrario, deja que V2 maneje 'atrás' (incluye devoluciones)
            return super().handle(user)
        if INTENTS["VIEW_CART"].search(u) or INTENTS["PAY"].search(u) or INTENTS["SEARCH"].search(u):
            # cambiar de operación en cualquier momento
            return super().handle(user)

        # Entrada rápida al flujo de reembolso desde cualquier estado
        if INTENTS["REFUND_STATUS"].search(u) and self.ctx.state != "REFUND_INFO":
            self.ctx.state = "REFUND_INFO"; return self.prompt()

        # Estado REFUND_INFO: pedir número de pedido o 'el más reciente'
        if self.ctx.state == "REFUND_INFO":
            m = INTENTS["ORDER_ID_OR_RECENT"].search(u)
            if not m:
                ods = list_orders()
                if not ods:
                    self.ctx.state = "INICIO"; return "No hay pedidos registrados."
                lines = ["Pedidos (más recientes primero):"]
                for i,o in enumerate(ods,1):
                    lines.append(f"{i}) {o['order_number']} — ${o['total_cents']/100:.2f}")
                lines.append("Dime el número de pedido o 'el más reciente'.")
                return "\n".join(lines)
            key = m.group(1)
            order_num = None if ("reciente" in key or "ultimo" in key or "último" in key) else key
            msg = refund_status(order_num)
            self.ctx.state = "INICIO"
            return msg + "\n\n" + super().prompt()

        # Si no estamos en REFUND_INFO, delegamos al comportamiento de devoluciones/compras
        return super().handle(user)

print("✅ PASO 7 listo — Reembolsos integrados (AmazonBotV3)")

✅ PASO 7 listo — Reembolsos integrados (AmazonBotV3)


In [8]:
# PASO 8 — Administración: Direcciones y Métodos de Pago
import re
from dataclasses import dataclass, field

# ---------- helpers de usuario ----------
def load_user() -> Dict[str, Any]:
    return read_json(user_path, {"user_id": 1, "addresses": [], "payment_methods": []})

def save_user(u: Dict[str, Any]):
    write_json(user_path, u)

def address_book_text(u: Dict[str, Any]) -> str:
    if not u["addresses"]:
        return "No tienes direcciones guardadas."
    lines = ["Direcciones:"]
    for i, a in enumerate(u["addresses"], 1):
        tag = " (predeterminada)" if a.get("is_default") else ""
        lines.append(f"{i}) {a['name']} — {a['street']}, {a['city']}, {a['state']}, CP {a['postal_code']} — {a['phone']}{tag}")
    return "\n".join(lines)

def cards_text(u: Dict[str, Any]) -> str:
    if not u["payment_methods"]:
        return "No tienes métodos de pago guardados."
    lines = ["Métodos de pago:"]
    for i, pm in enumerate(u["payment_methods"], 1):
        tag = " (predeterminado)" if pm.get("is_default") else ""
        brand = pm.get("brand","").upper()
        lines.append(f"{i}) {brand} ••{pm['last4']}{tag}")
    return "\n".join(lines)

def set_default_address(idx: int) -> str:
    u = load_user()
    if not (1 <= idx <= len(u["addresses"])): return "Índice inválido."
    for a in u["addresses"]: a["is_default"] = False
    u["addresses"][idx-1]["is_default"] = True
    save_user(u); return "Dirección marcada como predeterminada."

def delete_address(idx: int) -> str:
    u = load_user()
    if not (1 <= idx <= len(u["addresses"])): return "Índice inválido."
    removed = u["addresses"].pop(idx-1)
    # si quitaste la predeterminada, marca la 1ª (si existe)
    if removed.get("is_default") and u["addresses"]:
        u["addresses"][0]["is_default"] = True
    save_user(u); return "Dirección eliminada."

def add_address(name:str, phone:str, street:str, city:str, state_s:str, postal:str) -> str:
    # validaciones mini
    if not re.fullmatch(r"\d{10}", re.sub(r"\D", "", phone or "")): return "Teléfono inválido (se esperan 10 dígitos)."
    if not re.fullmatch(r"\d{5}", postal or ""): return "CP inválido (5 dígitos)."
    u = load_user()
    next_id = (max([a["id"] for a in u["addresses"]] or [0]) + 1)
    is_default = not any(a.get("is_default") for a in u["addresses"])
    u["addresses"].append({
        "id": next_id, "name": name, "phone": re.sub(r"\D","",phone),
        "street": street, "city": city, "state": state_s, "postal_code": postal,
        "is_default": is_default
    })
    save_user(u); 
    return "Dirección agregada."

def set_default_card(idx: int) -> str:
    u = load_user()
    if not (1 <= idx <= len(u["payment_methods"])): return "Índice inválido."
    for p in u["payment_methods"]: p["is_default"] = False
    u["payment_methods"][idx-1]["is_default"] = True
    save_user(u); return "Método de pago marcado como predeterminado."

def delete_card(idx: int) -> str:
    u = load_user()
    if not (1 <= idx <= len(u["payment_methods"])): return "Índice inválido."
    removed = u["payment_methods"].pop(idx-1)
    if removed.get("is_default") and u["payment_methods"]:
        u["payment_methods"][0]["is_default"] = True
    save_user(u); return "Método de pago eliminado."

def add_card(brand:str, last4:str) -> str:
    brand = normalize(brand)
    brand_map = {
        "visa":"visa", "mastercard":"mastercard", "master card":"mastercard",
        "american express":"american express", "amex":"american express", "carnet":"carnet"
    }
    # mapear marca
    found = None
    for k,v in brand_map.items():
        if k in brand: found = v; break
    if not found:
        return "Marca no reconocida. Usa: Visa, MasterCard, American Express o Carnet."
    if not re.fullmatch(r"\d{4}", last4 or ""):
        return "Los últimos 4 dígitos deben ser 4 números."
    u = load_user()
    next_id = (max([p["id"] for p in u["payment_methods"]] or [0]) + 1)
    is_default = not any(p.get("is_default") for p in u["payment_methods"])
    u["payment_methods"].append({"id":next_id, "brand":found, "last4":last4, "token":f"tok_{found}_{last4}", "is_default":is_default})
    save_user(u); 
    return "Método de pago agregado."

# ---------- intents extra para administración ----------
INTENTS.update({
    "ADMIN_DIR_ENTER": re.compile(r"\b(direcciones?|domicilios?)\b", re.I),
    "ADMIN_PAGO_ENTER": re.compile(r"\b(pagos?|tarjetas?|metodo de pago|m[eé]todo de pago)\b", re.I),
    "ADD_WORD":   re.compile(r"\b(agregar|añadir|anadir|nueva|registrar|dar de alta)\b", re.I),
    "DEL_IDX":    re.compile(r"\b(elimina(?:r)?|quitar|borrar)\b.*?(?P<idx>\d+)", re.I),
    "SETDEF_IDX": re.compile(r"\b(predeterminad[ao]|principal|por defecto|usar|poner(?:la)?\s+como\s+predeterminad[ao])\b.*?(?P<idx>\d+)", re.I),
    "BRAND":      re.compile(r"\b(visa|master ?card|american express|amex|carnet)\b", re.I),
    "LAST4":      re.compile(r"\b(\d{4})\b"),
})

# ---------- contexto para administración ----------
@dataclass
class AdminCtx:
    state: str = ""           # "", DIR_MENU, DIR_ADD_, DIR_DELETE, DIR_SETDEF, PAGO_MENU, PAGO_ADD_, PAGO_DELETE, PAGO_SETDEF
    # buffer: address
    a_name: Optional[str] = None
    a_phone: Optional[str] = None
    a_street: Optional[str] = None
    a_city: Optional[str] = None
    a_state: Optional[str] = None
    a_cp: Optional[str] = None
    # buffer: card
    c_brand: Optional[str] = None
    c_last4: Optional[str] = None

# ---------- clase principal con admin (extiende V3) ----------
class AmazonBotV4(AmazonBotV3):
    def __init__(self):
        super().__init__()
        self.admin = AdminCtx()

    # prompts específicos
    def prompt(self) -> str:
        s = self.admin.state
        if s == "DIR_MENU":
            u = load_user()
            return (address_book_text(u) + 
                    "\n\nOpciones: 'agregar', 'elimina 2', 'predeterminada 1', 'atrás' o 'salir'.")
        if s == "DIR_ADD_NAME":   return "Nombre de la persona receptora:"
        if s == "DIR_ADD_PHONE":  return "Teléfono (10 dígitos):"
        if s == "DIR_ADD_STREET": return "Calle y número:"
        if s == "DIR_ADD_CITY":   return "Ciudad/municipio:"
        if s == "DIR_ADD_STATE":  return "Estado:"
        if s == "DIR_ADD_CP":     return "Código Postal (5 dígitos):"
        if s == "PAGO_MENU":
            u = load_user()
            return (cards_text(u) + 
                    "\n\nOpciones: 'agregar', 'elimina 2', 'predeterminado 1', 'atrás' o 'salir'.")
        if s == "PAGO_ADD_BRAND": return "Marca de la tarjeta (Visa, MasterCard, American Express, Carnet):"
        if s == "PAGO_ADD_LAST4": return "Últimos 4 dígitos de la tarjeta:"
        # si no estamos en admin, delega al resto
        return super().prompt()

    def _to_root(self):
        # salir de admin a INICIO
        self.admin = AdminCtx()
        self.ctx.state = "INICIO"

    def handle(self, user: str) -> str:
        u = normalize(user)

        # filtros globales — mismos que ya tienes
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Quieres administrar direcciones/pagos o seguimos comprando?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("Puedo comprar, ver/editar carrito, pagar, devolver, ver reembolsos *y administrar direcciones/pagos*.")
            return ("Creo que eso es otro tema 😅. ¿Seguimos con compras, devoluciones o administración?")

        # comandos globales
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            # si estamos dentro de admin, regresar al menú de ese módulo o a INICIO
            if self.admin.state.startswith("DIR_"):
                # si está en algún paso de agregar, vuelve al menú
                self.admin.state = "DIR_MENU"; return self.prompt()
            if self.admin.state.startswith("PAGO_"):
                self.admin.state = "PAGO_MENU"; return self.prompt()
            # fuera de admin: comportamiento normal
            return super().handle(user)

        # cambiar de operación a cualquier otro flujo (compras/devoluciones/reembolso)
        if INTENTS["VIEW_CART"].search(u) or INTENTS["PAY"].search(u) or INTENTS["SEARCH"].search(u) or INTENTS["DEV_START"].search(u) or INTENTS["REFUND_STATUS"].search(u):
            self._to_root()
            return super().handle(user)

        # entrada a administración desde cualquier estado
        if INTENTS["ADMIN_DIR_ENTER"].search(u) and not self.admin.state:
            self.admin.state = "DIR_MENU"; return self.prompt()
        if INTENTS["ADMIN_PAGO_ENTER"].search(u) and not self.admin.state:
            self.admin.state = "PAGO_MENU"; return self.prompt()

        # ===== Direcciones =====
        if self.admin.state.startswith("DIR_"):
            s = self.admin.state
            if s == "DIR_MENU":
                if INTENTS["ADD_WORD"].search(u):
                    self.admin.state = "DIR_ADD_NAME"; return self.prompt()
                m = INTENTS["DEL_IDX"].search(u)
                if m:
                    idx = int(m.group("idx")); msg = delete_address(idx)
                    return msg + "\n\n" + self.prompt()
                m = INTENTS["SETDEF_IDX"].search(u)
                if m:
                    idx = int(m.group("idx")); msg = set_default_address(idx)
                    return msg + "\n\n" + self.prompt()
                # si escribe "pagos" o "direcciones" otra vez
                if INTENTS["ADMIN_PAGO_ENTER"].search(u):
                    self.admin.state = "PAGO_MENU"; return self.prompt()
                if INTENTS["ADMIN_DIR_ENTER"].search(u):
                    return self.prompt()
                return "No entendí. Opciones: 'agregar', 'elimina 2', 'predeterminada 1', 'atrás'."

            if s == "DIR_ADD_NAME":
                self.admin.a_name = user.strip() if user.strip() else None
                if not self.admin.a_name: return "Dame un nombre válido:"
                self.admin.state = "DIR_ADD_PHONE"; return self.prompt()

            if s == "DIR_ADD_PHONE":
                digits = re.sub(r"\D","", user)
                if not re.fullmatch(r"\d{10}", digits): return "Teléfono inválido. Debe tener 10 dígitos:"
                self.admin.a_phone = digits
                self.admin.state = "DIR_ADD_STREET"; return self.prompt()

            if s == "DIR_ADD_STREET":
                self.admin.a_street = user.strip()
                if not self.admin.a_street: return "Calle y número, por favor:"
                self.admin.state = "DIR_ADD_CITY"; return self.prompt()

            if s == "DIR_ADD_CITY":
                self.admin.a_city = user.strip()
                if not self.admin.a_city: return "Ciudad/municipio, por favor:"
                self.admin.state = "DIR_ADD_STATE"; return self.prompt()

            if s == "DIR_ADD_STATE":
                self.admin.a_state = user.strip()
                if not self.admin.a_state: return "Estado, por favor:"
                self.admin.state = "DIR_ADD_CP"; return self.prompt()

            if s == "DIR_ADD_CP":
                cp = user.strip()
                if not re.fullmatch(r"\d{5}", cp): return "CP inválido. Debe tener 5 dígitos:"
                self.admin.a_cp = cp
                # intentar guardar
                msg = add_address(self.admin.a_name, self.admin.a_phone, self.admin.a_street, self.admin.a_city, self.admin.a_state, self.admin.a_cp)
                # limpiar buffer y volver a menú
                self.admin = AdminCtx(state="DIR_MENU")
                return msg + "\n\n" + self.prompt()

        # ===== Pagos =====
        if self.admin.state.startswith("PAGO_"):
            s = self.admin.state
            if s == "PAGO_MENU":
                if INTENTS["ADD_WORD"].search(u):
                    self.admin.state = "PAGO_ADD_BRAND"; return self.prompt()
                m = INTENTS["DEL_IDX"].search(u)
                if m:
                    idx = int(m.group("idx")); msg = delete_card(idx)
                    return msg + "\n\n" + self.prompt()
                m = INTENTS["SETDEF_IDX"].search(u)
                if m:
                    idx = int(m.group("idx")); msg = set_default_card(idx)
                    return msg + "\n\n" + self.prompt()
                if INTENTS["ADMIN_DIR_ENTER"].search(u):
                    self.admin.state = "DIR_MENU"; return self.prompt()
                if INTENTS["ADMIN_PAGO_ENTER"].search(u):
                    return self.prompt()
                return "No entendí. Opciones: 'agregar', 'elimina 2', 'predeterminado 1', 'atrás'."

            if s == "PAGO_ADD_BRAND":
                m = INTENTS["BRAND"].search(u)
                if not m: return "Marca no reconocida. Usa: Visa, MasterCard, American Express o Carnet."
                self.admin.c_brand = m.group(1)
                self.admin.state = "PAGO_ADD_LAST4"; return self.prompt()

            if s == "PAGO_ADD_LAST4":
                m = INTENTS["LAST4"].search(u)
                if not m or not re.fullmatch(r"\d{4}", m.group(1)):
                    return "Los últimos 4 dígitos deben ser 4 números:"
                self.admin.c_last4 = m.group(1)
                msg = add_card(self.admin.c_brand, self.admin.c_last4)
                self.admin = AdminCtx(state="PAGO_MENU")
                return msg + "\n\n" + self.prompt()

        # si no estamos en admin, delega a devoluciones/reembolsos/compras
        return super().handle(user)

print("✅ PASO 8 listo — Administración integrada (AmazonBotV4)")

✅ PASO 8 listo — Administración integrada (AmazonBotV4)


In [9]:
# PASO 9 — Checkout con confirmación y método de pago (AmazonBotV5)

from dataclasses import dataclass, field

# --- Intents extra específicos del checkout ---
INTENTS.update({
    # "usar 2", "método 1", "tarjeta 3"
    "PAY_USE_INDEX": re.compile(r"\b(?:usar|metod[oó]|m[eé]todo|tarjeta)\s+(?P<idx>\d+)\b", re.I),
    # "cambiar método", "otra tarjeta"
    "PAY_CHANGE":    re.compile(r"\b(cambiar\s+(?:metod[oó]|m[eé]todo|tarjeta)|otr[ao]\s+(?:metod[oó]|m[eé]todo|pago|tarjeta))\b", re.I),
})

# --- Helpers de checkout ---
def default_address() -> Optional[Dict[str, Any]]:
    u = load_user()
    if not u["addresses"]: return None
    for a in u["addresses"]:
        if a.get("is_default"): return a
    return u["addresses"][0]

def list_cards() -> List[Dict[str, Any]]:
    return load_user().get("payment_methods", [])

def format_address_line(a: Dict[str, Any]) -> str:
    return f"{a['name']} — {a['street']}, {a['city']}, {a['state']}, CP {a['postal_code']} — {a['phone']}"

def checkout_review_text(selected_idx: Optional[int]=None) -> str:
    """
    Arma el texto de revisión: carrito + dirección + métodos de pago.
    selected_idx: índice (1-based) de tarjeta seleccionada temporalmente en este checkout.
    """
    lines = ["RESUMEN DEL PEDIDO", "-------------------"]
    # Carrito + total
    lines.append(cart_summary())
    # Dirección
    a = default_address()
    lines.append("\nDirección de entrega:")
    lines.append(format_address_line(a) if a else " (sin dirección — usa 'direcciones' para agregar una)")
    # Pago
    cards = list_cards()
    if not cards:
        lines.append("\nMétodo de pago: (no tienes tarjetas guardadas)")
        lines.append("— Escribe 'agregar' para registrar una tarjeta.")
    else:
        lines.append("\nMétodo de pago:")
        for i, pm in enumerate(cards, 1):
            tag = []
            if pm.get("is_default"): tag.append("predeterminado")
            if selected_idx and selected_idx == i: tag.append("seleccionado")
            tag_txt = f" ({', '.join(tag)})" if tag else ""
            lines.append(f"{i}) {pm['brand'].upper()} ••{pm['last4']}{tag_txt}")
        lines.append("\nOpciones: 'usar 2', 'cambiar método', 'agregar', 'confirmar', 'atrás', 'salir'.")
    return "\n".join(lines)

@dataclass
class CheckoutCtx:
    state: str = ""              # CHK_REVIEW, CHK_SELECT_PAY, CHK_ADD_BRAND, CHK_ADD_LAST4
    selected_card_idx: Optional[int] = None
    add_brand: Optional[str] = None

class AmazonBotV5(AmazonBotV4):
    def __init__(self):
        super().__init__()
        self.chk = CheckoutCtx()

    # --- Prompt extendido para los estados de checkout ---
    def prompt(self) -> str:
        s = self.chk.state
        if s == "CHK_REVIEW":
            return checkout_review_text(self.chk.selected_card_idx)
        if s == "CHK_SELECT_PAY":
            u = load_user()
            txt = cards_text(u)
            if "No tienes métodos" in txt:
                txt += "\n\nEscribe 'agregar' para registrar una tarjeta."
            else:
                txt += "\n\nDi 'usar 2' para elegir temporalmente durante este pago, o 'agregar'."
            return txt
        if s == "CHK_ADD_BRAND":
            return "Marca de la tarjeta (Visa, MasterCard, American Express, Carnet):"
        if s == "CHK_ADD_LAST4":
            return "Últimos 4 dígitos de la tarjeta:"
        # estados no-CHK delegan al prompt anterior
        return super().prompt()

    def _exit_checkout(self):
        self.chk = CheckoutCtx()
        # al salir del checkout, volvemos a INICIO
        self.ctx.state = "INICIO"

    def handle(self, user: str) -> str:
        u = normalize(user)

        # Filtros globales (ofensivo / off-topic)
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Seguimos con la compra o prefieres salir?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("En checkout puedo: 'usar 2', 'cambiar método', 'agregar', 'confirmar', 'atrás' o 'salir'.")
            return ("Creo que eso es otro tema 😅. ¿Quieres confirmar, cambiar método o agregar tarjeta?")
        self.offtopic_count = 0

        # Salir / Atrás (globales)
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            # Si estamos dentro del flujo de checkout, regresar a CARRITO
            if self.chk.state:
                self.chk = CheckoutCtx()
                self.ctx.state = "CARRITO"
                return self.prompt()
            # fuera del checkout: comportamiento estándar
            return super().handle(user)

        # Cambiar de operación en cualquier momento (ver carrito, buscar, devolver, reembolso, admin)
        if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or
            INTENTS["DEV_START"].search(u) or INTENTS["REFUND_STATUS"].search(u) or
            INTENTS["ADMIN_DIR_ENTER"].search(u) or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
            # Sal del checkout si estabas en él
            if self.chk.state:
                self.chk = CheckoutCtx()
            return super().handle(user)

        # --- Interceptar "pagar" para iniciar checkout con confirmación ---
        if INTENTS["PAY"].search(u) and not self.chk.state:
            cart = load_cart()
            if not cart["items"]:
                # Igual que antes, sin iniciar flujo
                return super().handle(user)
            # Inicia flujo de checkout
            self.chk = CheckoutCtx(state="CHK_REVIEW", selected_card_idx=None)
            return self.prompt()

        # ===== Flujo de Checkout (CHK_*) =====
        s = self.chk.state

        if s == "CHK_REVIEW":
            # Confirmar pedido
            if INTENTS["CONFIRM"].search(u):
                cards = list_cards()
                # si no hay tarjetas, fuerza agregar
                if not cards:
                    self.chk.state = "CHK_ADD_BRAND"
                    return "No tienes tarjetas guardadas. Vamos a agregar una.\n" + self.prompt()
                # elegir tarjeta efectiva: la seleccionada temporal o la predeterminada
                if self.chk.selected_card_idx:
                    if not (1 <= self.chk.selected_card_idx <= len(cards)):
                        return "Índice de tarjeta inválido. Usa 'usar 1' o 'agregar'."
                    card = cards[self.chk.selected_card_idx - 1]
                else:
                    # por defecto: la predeterminada o la 1ª
                    card = next((c for c in cards if c.get("is_default")), cards[0])

                # Finaliza el pedido con la selección
                msg = checkout()
                self._exit_checkout()
                return msg + f"\nPagado con {card['brand'].upper()} ••{card['last4']}.\n\n" + self.prompt()

            # Cambiar/seleccionar método de pago
            if INTENTS["PAY_CHANGE"].search(u):
                self.chk.state = "CHK_SELECT_PAY"
                return self.prompt()

            m = INTENTS["PAY_USE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                if not (1 <= idx <= len(list_cards())):
                    return "Índice de tarjeta inválido. Prueba con 'usar 1' o 'agregar'."
                self.chk.selected_card_idx = idx
                return checkout_review_text(self.chk.selected_card_idx)

            # Agregar tarjeta nueva desde la revisión
            if INTENTS["ADD_WORD"].search(u):
                self.chk.state = "CHK_ADD_BRAND"
                return self.prompt()

            # Si escribe "pagar" otra vez aquí, trátalo como confirmar
            if INTENTS["PAY"].search(u):
                return "Si ya revisaste, escribe *'confirmar'* para completar tu pedido, o 'cambiar método'/'agregar'."

            return self.prompt()

        if s == "CHK_SELECT_PAY":
            # Seleccionar por índice
            m = INTENTS["PAY_USE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                if not (1 <= idx <= len(list_cards())):
                    return "Índice inválido. Di 'usar 1' o 'agregar'."
                self.chk.selected_card_idx = idx
                self.chk.state = "CHK_REVIEW"
                return self.prompt()
            # Agregar uno nuevo
            if INTENTS["ADD_WORD"].search(u):
                self.chk.state = "CHK_ADD_BRAND"
                return self.prompt()
            # Cualquier otra cosa: repetir menú
            return self.prompt()

        if s == "CHK_ADD_BRAND":
            m = INTENTS["BRAND"].search(u)
            if not m:
                return "Marca no reconocida. Usa: Visa, MasterCard, American Express o Carnet."
            self.chk.add_brand = m.group(1)
            self.chk.state = "CHK_ADD_LAST4"
            return self.prompt()

        if s == "CHK_ADD_LAST4":
            m = INTENTS["LAST4"].search(u)
            if not m or not re.fullmatch(r"\d{4}", m.group(1)):
                return "Los últimos 4 dígitos deben ser 4 números:"
            last4 = m.group(1)
            # agregamos la tarjeta (no la marcamos predeterminada a menos que sea la primera)
            msg = add_card(self.chk.add_brand, last4)
            # la tarjeta recién agregada es la última: selección temporal
            self.chk.selected_card_idx = len(list_cards())
            self.chk.state = "CHK_REVIEW"
            return msg + "\n\n" + self.prompt()

        # fuera del flujo de checkout, delega al comportamiento previo
        return super().handle(user)

print("✅ PASO 9 listo — Checkout con revisión + selección/agregado de método de pago (AmazonBotV5)")

✅ PASO 9 listo — Checkout con revisión + selección/agregado de método de pago (AmazonBotV5)


In [10]:
# PASO 10 — AmazonBotV6: checkout con cambio/alta de DIRECCIÓN y comandos "proceder al pago" + "realiza tu pedido y paga"

import re
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

# --- Intents nuevos específicos del checkout ---
INTENTS.update({
    # Sinónimos para iniciar el flujo de revisión
    "PROCEED_PAY": re.compile(r"\b(proceder (?:al|a) pago|continuar (?:al|con el) pago|ir a pagar)\b", re.I),
    # Confirmación final estilo Amazon
    "CONFIRM_PAY": re.compile(r"\b(realiza(?:r)? tu pedido(?: y paga)?|realiza tu pedido y paga|pagar ahora)\b", re.I),

    # Dirección: cambiar/usar/agregar dentro del checkout
    "PAY_ADDR_CHANGE": re.compile(r"\b(cambiar\s+(?:direccion|dirección)|otra\s+(?:direccion|dirección))\b", re.I),
    "PAY_ADDR_USE_INDEX": re.compile(r"\b(?:usar|direccion|dirección)\s+(?P<idx>\d+)\b", re.I),
    "ADDR_WORD": re.compile(r"\b(direccion|dirección|direcciones)\b", re.I),
})

def list_addresses() -> List[Dict[str, Any]]:
    return load_user().get("addresses", [])

# --- extendemos el contexto de checkout con selección de dirección ---
@dataclass
class CheckoutCtxV6(CheckoutCtx):
    selected_addr_idx: Optional[int] = None
    # buffers para alta de dirección dentro del checkout
    a_name: Optional[str] = None
    a_phone: Optional[str] = None
    a_street: Optional[str] = None
    a_city: Optional[str] = None
    a_state: Optional[str] = None
    a_cp: Optional[str] = None

class AmazonBotV6(AmazonBotV5):
    def __init__(self):
        super().__init__()
        self.chk = CheckoutCtxV6()

    # Texto de revisión con dirección y pago, respetando selecciones temporales
    def _review_text(self) -> str:
        lines = ["RESUMEN DEL PEDIDO", "-------------------"]
        lines.append(cart_summary())

        # Dirección efectiva: seleccionada temporalmente o predeterminada
        addrs = list_addresses()
        eff_addr = None
        if addrs:
            if self.chk.selected_addr_idx and 1 <= self.chk.selected_addr_idx <= len(addrs):
                eff_addr = addrs[self.chk.selected_addr_idx-1]
            else:
                eff_addr = next((a for a in addrs if a.get("is_default")), addrs[0])

        lines.append("\nDirección de entrega:")
        if eff_addr:
            lines.append(format_address_line(eff_addr))
        else:
            lines.append("(sin dirección — escribe 'agregar dirección' para registrar una)")

        # Métodos de pago
        cards = list_cards()
        if not cards:
            lines.append("\nMétodo de pago: (no tienes tarjetas guardadas)")
            lines.append("— Escribe 'agregar tarjeta' para registrar una.")
        else:
            lines.append("\nMétodo de pago:")
            for i, pm in enumerate(cards, 1):
                tag = []
                if pm.get("is_default"): tag.append("predeterminado")
                if self.chk.selected_card_idx and self.chk.selected_card_idx == i: tag.append("seleccionado")
                lines.append(f"{i}) {pm['brand'].upper()} ••{pm['last4']}" + (f" ({', '.join(tag)})" if tag else ""))

        # Ayuda de acciones
        lines.append(
            "\nOpciones:"
            " 'usar 2' (tarjeta), 'cambiar método', 'agregar tarjeta',"
            " 'usar dirección 2', 'cambiar dirección', 'agregar dirección',"
            " 'confirmar' / 'realiza tu pedido y paga', 'atrás', 'salir'."
        )
        return "\n".join(lines)

    # prompts de estados nuevos/extendidos
    def prompt(self) -> str:
        s = self.chk.state

        # Revisión
        if s == "CHK_REVIEW":
            return self._review_text()

        # Selección de pago ya existe (V5): CHK_SELECT_PAY, CHK_ADD_BRAND, CHK_ADD_LAST4

        # Menú de direcciones dentro del checkout
        if s == "CHK_ADDR_MENU":
            txt = address_book_text(load_user())
            if "No tienes direcciones" in txt:
                txt += "\n\nEscribe 'agregar dirección' para registrar una."
            else:
                txt += "\n\nDi 'usar dirección 2' para elegir temporalmente durante este pago, o 'agregar dirección'."
            return txt

        # Flujo de alta de dirección (igual que admin, pero local al checkout)
        if s == "CHK_ADDR_ADD_NAME":   return "Nombre de la persona receptora:"
        if s == "CHK_ADDR_ADD_PHONE":  return "Teléfono (10 dígitos):"
        if s == "CHK_ADDR_ADD_STREET": return "Calle y número:"
        if s == "CHK_ADDR_ADD_CITY":   return "Ciudad/municipio:"
        if s == "CHK_ADDR_ADD_STATE":  return "Estado:"
        if s == "CHK_ADDR_ADD_CP":     return "Código Postal (5 dígitos):"

        return super().prompt()

    def _enter_checkout_review(self) -> str:
        self.chk.state = "CHK_REVIEW"
        return self.prompt()

    def _exit_checkout(self):
        self.chk = CheckoutCtxV6()
        self.ctx.state = "INICIO"

    def handle(self, user: str) -> str:
        u = normalize(user)

        # Filtros globales
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor. ¿Seguimos con el pago o prefieres salir?"
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("En checkout puedes: cambiar/usar dirección, cambiar/usar método, agregar dirección/tarjeta, confirmar.")
            return ("¿Quieres 'cambiar dirección', 'cambiar método' o 'confirmar'?")
        self.offtopic_count = 0

        # Salir / Atrás
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            if self.chk.state:
                # Regresar al carrito desde cualquier estado CHK_*
                self.chk = CheckoutCtxV6()
                self.ctx.state = "CARRITO"
                return self.prompt()
            return super().handle(user)

        # Cambiar de operación fuera del checkout
        if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or
            INTENTS["DEV_START"].search(u) or INTENTS["REFUND_STATUS"].search(u) or
            INTENTS["ADMIN_DIR_ENTER"].search(u) or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
            if self.chk.state:
                self.chk = CheckoutCtxV6()
            return super().handle(user)

        # Iniciar revisión por "pagar" o "proceder al pago"
        if (INTENTS["PAY"].search(u) or INTENTS["PROCEED_PAY"].search(u)) and not self.chk.state:
            cart = load_cart()
            if not cart["items"]:
                return super().handle(user)  # sin items, delega
            return self._enter_checkout_review()

        # ====== Flujo CHK_* ======
        s = self.chk.state

        # Revisión
        if s == "CHK_REVIEW":
            # Confirmación final (sí/confirmar o frases largas estilo Amazon)
            if INTENTS["CONFIRM"].search(u) or INTENTS["CONFIRM_PAY"].search(u):
                addrs = list_addresses()
                if not addrs:
                    # Forzar alta de dirección
                    self.chk.state = "CHK_ADDR_ADD_NAME"
                    return "No tienes dirección registrada. Vamos a agregar una.\n" + self.prompt()
                cards = list_cards()
                if not cards:
                    # Forzar alta de tarjeta (estado V5)
                    self.chk.state = "CHK_ADD_BRAND"
                    return "No tienes tarjetas guardadas. Vamos a agregar una.\n" + self.prompt()

                # Dirección efectiva
                if self.chk.selected_addr_idx and 1 <= self.chk.selected_addr_idx <= len(addrs):
                    addr = addrs[self.chk.selected_addr_idx-1]
                else:
                    addr = next((a for a in addrs if a.get("is_default")), addrs[0])

                # Tarjeta efectiva
                if self.chk.selected_card_idx and 1 <= self.chk.selected_card_idx <= len(cards):
                    card = cards[self.chk.selected_card_idx - 1]
                else:
                    card = next((c for c in cards if c.get("is_default")), cards[0])

                # Ejecutar cobro/checkout
                msg = checkout()
                self._exit_checkout()
                return msg + f"\nEnviaremos a: {format_address_line(addr)}\nPagado con {card['brand'].upper()} ••{card['last4']}.\n\n" + self.prompt()

            # Cambiar/usar MÉTODO de pago
            if INTENTS["PAY_CHANGE"].search(u):
                self.chk.state = "CHK_SELECT_PAY"
                return self.prompt()
            m = INTENTS["PAY_USE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                if not (1 <= idx <= len(list_cards())):
                    return "Índice de tarjeta inválido. Prueba con 'usar 1' o 'agregar tarjeta'."
                self.chk.selected_card_idx = idx
                return self._review_text()

            # Cambiar/usar DIRECCIÓN
            if INTENTS["PAY_ADDR_CHANGE"].search(u):
                self.chk.state = "CHK_ADDR_MENU"
                return self.prompt()
            m = INTENTS["PAY_ADDR_USE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                if not (1 <= idx <= len(list_addresses())):
                    return "Índice de dirección inválido. Prueba con 'usar dirección 1' o 'agregar dirección'."
                self.chk.selected_addr_idx = idx
                return self._review_text()

            # Agregar explícitamente tarjeta o dirección
            if INTENTS["ADD_WORD"].search(u) and INTENTS["ADDR_WORD"].search(u):
                self.chk.state = "CHK_ADDR_ADD_NAME"
                return self.prompt()
            if INTENTS["ADD_WORD"].search(u) and INTENTS["BRAND"].search(u):
                self.chk.state = "CHK_ADD_BRAND"
                return self.prompt()
            if INTENTS["ADD_WORD"].search(u):
                return "¿Quieres *agregar tarjeta* o *agregar dirección*?"

            # Repetir la revisión por defecto
            if INTENTS["PAY"].search(u):
                return "Para finalizar, escribe *'confirmar'* o *'realiza tu pedido y paga'*."
            return self._review_text()

        # Selección de pago y alta de tarjeta (ya implementados en V5)
        if s in ("CHK_SELECT_PAY","CHK_ADD_BRAND","CHK_ADD_LAST4"):
            return super().handle(user)

        # Menú de direcciones en checkout
        if s == "CHK_ADDR_MENU":
            # usar por índice
            m = INTENTS["PAY_ADDR_USE_INDEX"].search(u)
            if m:
                idx = int(m.group("idx"))
                if not (1 <= idx <= len(list_addresses())):
                    return "Índice inválido. Di 'usar dirección 1' o 'agregar dirección'."
                self.chk.selected_addr_idx = idx
                self.chk.state = "CHK_REVIEW"
                return self.prompt()
            # agregar
            if INTENTS["ADD_WORD"].search(u) and INTENTS["ADDR_WORD"].search(u):
                self.chk.state = "CHK_ADDR_ADD_NAME"
                return self.prompt()
            if INTENTS["ADD_WORD"].search(u):
                # si solo puso "agregar"
                return "¿Agregar *dirección*? Di 'agregar dirección' o 'usar dirección 1'."
            # set predeterminada / eliminar (reutiliza intents globales de admin)
            m = INTENTS["SETDEF_IDX"].search(u)
            if m:
                idx = int(m.group("idx"))
                msg = set_default_address(idx)
                return msg + "\n\n" + self.prompt()
            m = INTENTS["DEL_IDX"].search(u)
            if m:
                idx = int(m.group("idx"))
                msg = delete_address(idx)
                return msg + "\n\n" + self.prompt()
            return self.prompt()

        # Alta de dirección dentro del checkout (mismo flujo que admin, pero local)
        if s == "CHK_ADDR_ADD_NAME":
            name = user.strip()
            if not name: return "Nombre de la persona receptora:"
            self.chk.a_name = name
            self.chk.state = "CHK_ADDR_ADD_PHONE"; return self.prompt()

        if s == "CHK_ADDR_ADD_PHONE":
            digits = re.sub(r"\D","", user)
            if not re.fullmatch(r"\d{10}", digits): return "Teléfono inválido. Debe tener 10 dígitos:"
            self.chk.a_phone = digits
            self.chk.state = "CHK_ADDR_ADD_STREET"; return self.prompt()

        if s == "CHK_ADDR_ADD_STREET":
            street = user.strip()
            if not street: return "Calle y número, por favor:"
            self.chk.a_street = street
            self.chk.state = "CHK_ADDR_ADD_CITY"; return self.prompt()

        if s == "CHK_ADDR_ADD_CITY":
            city = user.strip()
            if not city: return "Ciudad/municipio, por favor:"
            self.chk.a_city = city
            self.chk.state = "CHK_ADDR_ADD_STATE"; return self.prompt()

        if s == "CHK_ADDR_ADD_STATE":
            st = user.strip()
            if not st: return "Estado, por favor:"
            self.chk.a_state = st
            self.chk.state = "CHK_ADDR_ADD_CP"; return self.prompt()

        if s == "CHK_ADDR_ADD_CP":
            cp = user.strip()
            if not re.fullmatch(r"\d{5}", cp): return "CP inválido. Debe tener 5 dígitos:"
            self.chk.a_cp = cp
            # Guardar dirección
            msg = add_address(self.chk.a_name, self.chk.a_phone, self.chk.a_street, self.chk.a_city, self.chk.a_state, self.chk.a_cp)
            # Seleccionar temporalmente la nueva (última del listado)
            self.chk.selected_addr_idx = len(list_addresses())
            # Limpiar buffers de dirección y volver a revisión
            self.chk.a_name = self.chk.a_phone = self.chk.a_street = self.chk.a_city = self.chk.a_state = self.chk.a_cp = None
            self.chk.state = "CHK_REVIEW"
            return msg + "\n\n" + self._review_text()

        # Si no estás en checkout aquí, delega al comportamiento previo
        return super().handle(user)

print("✅ PASO 10 listo — Checkout con cambio/alta de dirección y comandos 'proceder al pago'/'realiza tu pedido y paga' (AmazonBotV6)")

✅ PASO 10 listo — Checkout con cambio/alta de dirección y comandos 'proceder al pago'/'realiza tu pedido y paga' (AmazonBotV6)


In [11]:
# PASO 11 — Marcar como predeterminada (dirección o tarjeta) desde la revisión del checkout
import re

# Intents para marcar como predeterminada en la pantalla de revisión
INTENTS.update({
    "SETDEF_ADDR_REVIEW": re.compile(r"\b(?:poner|dejar|marcar)\s+(?:la\s+)?(?:direccion|dirección)\s+(?:como\s+)?predeterminad[ao]\b", re.I),
    "SETDEF_CARD_REVIEW": re.compile(r"\b(?:poner|dejar|marcar)\s+(?:la\s+)?(?:tarjeta|metod[oó]|m[eé]todo)\s+(?:como\s+)?predeterminad[ao]\b", re.I),
})

class AmazonBotV7(AmazonBotV6):
    def __init__(self):
        super().__init__()

    # helpers para obtener el índice efectivo (1-based) de dirección/tarjeta en la revisión
    def _effective_addr_index(self) -> Optional[int]:
        addrs = list_addresses()
        if not addrs:
            return None
        if self.chk.selected_addr_idx and 1 <= self.chk.selected_addr_idx <= len(addrs):
            return self.chk.selected_addr_idx
        # predeterminada o 1ª
        for i,a in enumerate(addrs,1):
            if a.get("is_default"):
                return i
        return 1

    def _effective_card_index(self) -> Optional[int]:
        cards = list_cards()
        if not cards:
            return None
        if self.chk.selected_card_idx and 1 <= self.chk.selected_card_idx <= len(cards):
            return self.chk.selected_card_idx
        for i,c in enumerate(cards,1):
            if c.get("is_default"):
                return i
        return 1

    def handle(self, user: str) -> str:
        u = normalize(user)

        # filtros globales usuales
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor."
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("En la revisión puedes: cambiar/usar dirección, cambiar/usar método, agregar dirección/tarjeta, "
                        "marcar como predeterminada, confirmar.")
            return ("¿Quieres cambiar dirección/método, marcar como predeterminada o confirmar?")
        self.offtopic_count = 0

        # salir / atrás / cambios de operación
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            if self.chk.state:
                self.chk = CheckoutCtxV6()
                self.ctx.state = "CARRITO"
                return self.prompt()
            return super().handle(user)
        if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or
            INTENTS["DEV_START"].search(u) or INTENTS["REFUND_STATUS"].search(u) or
            INTENTS["ADMIN_DIR_ENTER"].search(u) or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
            if self.chk.state:
                self.chk = CheckoutCtxV6()
            return super().handle(user)

        # iniciar revisión por "pagar" o "proceder al pago"
        if (INTENTS["PAY"].search(u) or INTENTS["PROCEED_PAY"].search(u)) and not self.chk.state:
            cart = load_cart()
            if not cart["items"]:
                return super().handle(user)
            return self._enter_checkout_review()

        # ===== lógica especial en la pantalla de revisión =====
        if self.chk.state == "CHK_REVIEW":
            # marcar dirección como predeterminada
            if INTENTS["SETDEF_ADDR_REVIEW"].search(u):
                idx = self._effective_addr_index()
                if not idx:
                    return "No tienes direcciones guardadas. Escribe 'agregar dirección'."
                msg = set_default_address(idx)
                return msg + "\n\n" + self._review_text()

            # marcar tarjeta como predeterminada
            if INTENTS["SETDEF_CARD_REVIEW"].search(u):
                idx = self._effective_card_index()
                if not idx:
                    return "No tienes tarjetas guardadas. Escribe 'agregar tarjeta'."
                msg = set_default_card(idx)
                return msg + "\n\n" + self._review_text()

            # el resto (confirmar, cambiar/usar dirección o método, agregar) ya lo maneja V6
            # delegamos para no duplicar lógica:
            return super().handle(user)

        # fuera de revisión/checkout, delega al comportamiento previo
        return super().handle(user)

print("✅ PASO 11 listo — Ahora puedes 'dejar como predeterminada' dirección o tarjeta desde la revisión (AmazonBotV7)")

✅ PASO 11 listo — Ahora puedes 'dejar como predeterminada' dirección o tarjeta desde la revisión (AmazonBotV7)


In [14]:
# PASO 12 — Cancelación de pedido + completar reembolso al recibir devolución (AmazonBotV8)

import csv, re
from datetime import datetime, timedelta
from dataclasses import dataclass

# ===== Configuración de la política demo =====
CANCEL_WINDOW_MIN = 30  # ventana para cancelar sin haber “enviado” (simulada)

# ===== Intents nuevos =====
INTENTS.update({
    "CANCEL_START": re.compile(r"\b(cancelar(?: pedido)?|cancelacion|cancelación|cancela lo|ya no lo quiero)\b", re.I),
    "RETURN_MARK_RECEIVED": re.compile(r"\b(marcar|ya) (?:devolucion|devolución) (?:#?)(?P<rid>\d+)? (?:recibida|entregada)\b", re.I),
})

# ===== Helpers de cancelación y reembolsos =====
def _orders_rows():
    with open(orders_path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _save_orders_rows(rows):
    with open(orders_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["order_number","status","subtotal_cents","shipping_cents","total_cents","created_at"])
        w.writeheader()
        for r in rows:
            w.writerow(r)

def _restock_on_cancel(order_number: str):
    """Regresa al inventario las piezas del pedido cancelado."""
    items = list_order_items(order_number)

    # 1) Variantes (ropa): sumar al CSV de variantes
    with open(variants_path, encoding="utf-8") as f:
        var_rows = list(csv.DictReader(f))
    for it in items:
        if it["variant_id"]:
            for row in var_rows:
                if int(row["variant_id"]) == it["variant_id"]:
                    row["stock"] = str(int(row["stock"]) + it["qty"])
    with open(variants_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["variant_id","product_id","size","stock"])
        w.writeheader()
        for r in var_rows:
            w.writerow(r)

    # 2) Productos sin talla: sumar al CSV de productos
    with open(products_path, encoding="utf-8") as f:
        prod_rows = list(csv.DictReader(f))
    for it in items:
        if not it["variant_id"]:
            for row in prod_rows:
                if int(row["product_id"]) == it["product_id"]:
                    row["stock"] = str(int(row["stock"]) + it["qty"])
    with open(products_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["product_id","name","category","price_cents","has_sizes","stock"])
        w.writeheader()
        for r in prod_rows:
            w.writerow(r)

    # refrescar caches
    load_products_if_changed()

def cancel_eligible(order_row: dict) -> bool:
    """Se puede cancelar si está 'paid' y dentro de CANCEL_WINDOW_MIN minutos."""
    if order_row["status"] != "paid":
        return False
    created = _parse_dt(order_row.get("created_at",""))
    return (datetime.now() - created) <= timedelta(minutes=CANCEL_WINDOW_MIN)

def _create_refund(order_number: str, amount_cents: int, status: str = "pending"):
    """Crea un registro de reembolso (p. ej., por cancelación total)."""
    # conseguir siguiente id
    next_id = 1
    with open(refunds_path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try: next_id = max(next_id, int(row["refund_id"]) + 1)
            except: pass
    processed = datetime.now().isoformat(timespec="seconds") if status == "completed" else ""
    with open(refunds_path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([next_id, order_number, int(amount_cents), status, processed])

def cancel_order(order_number: str) -> dict:
    """Cancela el pedido si es elegible, regresa stock y genera reembolso total (pendiente)."""
    rows = _orders_rows()
    row = next((r for r in rows if r["order_number"] == order_number), None)
    if not row:
        return {"ok": False, "msg": "No encontré ese pedido."}
    if row["status"] == "canceled":
        return {"ok": False, "msg": "Ese pedido ya está cancelado."}
    if not cancel_eligible(row):
        return {"ok": False, "msg": f"No se puede cancelar: fuera de la ventana de {CANCEL_WINDOW_MIN} min o ya procesado."}

    # Marcar cancelado
    row["status"] = "canceled"
    _save_orders_rows(rows)

    # Regresar stock
    _restock_on_cancel(order_number)

    # Reembolso total (pendiente)
    total = int(row["total_cents"])
    _create_refund(order_number, total, status="pending")

    return {"ok": True, "msg": f"Pedido {order_number} cancelado. Reembolso de ${total/100:.2f} en proceso."}

def mark_return_received(return_id: int) -> dict:
    """Marca la devolución como 'received' y completa el reembolso asociado (por monto)."""
    # 1) Buscar la devolución
    with open(returns_path, encoding="utf-8") as f:
        ret_rows = list(csv.DictReader(f))
    rr = next((r for r in ret_rows if r["return_id"] == str(return_id)), None)
    if not rr:
        return {"ok": False, "msg": f"No encuentro la devolución #{return_id}."}

    # 2) Actualizar estatus de la devolución
    rr["status"] = "received"
    with open(returns_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["return_id","order_number","item_idx","product_id","variant_id","size","qty","reason","method","status","created_at"])
        w.writeheader()
        for r in ret_rows:
            w.writerow(r)

    # 3) Calcular monto exacto de esa devolución
    order_number = rr["order_number"]
    item_idx = int(rr["item_idx"])
    qty = int(rr["qty"])
    items = list_order_items(order_number)
    if not (1 <= item_idx <= len(items)):
        return {"ok": False, "msg": "Índice de artículo inválido en la devolución."}
    it = items[item_idx-1]
    amount = it["unit_price_cents"] * qty

    # 4) Marcar como 'completed' el primer reembolso pendiente con ese monto
    with open(refunds_path, encoding="utf-8") as f:
        ref_rows = list(csv.DictReader(f))
    found = None
    for rf in ref_rows:
        if rf["order_number"] == order_number and int(rf["amount_cents"]) == amount and rf["status"] in ("", "pending"):
            rf["status"] = "completed"
            rf["processed_at"] = datetime.now().isoformat(timespec="seconds")
            found = rf
            break
    # Si no había reembolso pendiente igual, crea uno completado (consistencia demo)
    if not found:
        _create_refund(order_number, amount, status="completed")
    else:
        with open(refunds_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["refund_id","order_number","amount_cents","status","processed_at"])
            w.writeheader()
            for r in ref_rows:
                w.writerow(r)

    return {"ok": True, "msg": f"Devolución #{return_id} marcada como recibida. Reembolso de ${amount/100:.2f} completado."}

# ===== Clase con flujo de cancelación y 'recibir devolución' =====
class AmazonBotV8(AmazonBotV7):
    def __init__(self):
        super().__init__()
        # contexto mínimo para cancelación
        self.cancel_order_num = None

    def prompt(self) -> str:
        if self.ctx.state == "CANCEL_INICIO":
            return "¿Qué pedido deseas cancelar? Dime el número (123-1234567-1234567) o 'el más reciente'."
        if self.ctx.state == "CANCEL_CONFIRMAR":
            return f"Vas a cancelar el pedido {self.cancel_order_num}. ¿Confirmas? (sí/no)"
        return super().prompt()

    def handle(self, user: str) -> str:
        u = normalize(user)

        # Filtros globales
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor."
        if INTENTS["OFF_TOPIC"].search(u):
            self.offtopic_count += 1
            if self.offtopic_count >= 2:
                return ("Puedo comprar, ver/editar carrito, pagar, devolver, revisar reembolsos, administrar y *cancelar pedidos*.")
            return ("¿Deseas cancelar un pedido, devolver algo o seguir comprando?")
        self.offtopic_count = 0

        # Salir / Atrás
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["BACK"].search(u):
            if self.ctx.state in ("CANCEL_INICIO","CANCEL_CONFIRMAR"):
                self.ctx.state = "INICIO"; self.cancel_order_num = None
                return self.prompt()
            # checkout/admin/dev/ref: delega a las clases base
            return super().handle(user)

        # Cambiar de operación (compras, checkout, dev, refund, admin)
        if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or INTENTS["PAY"].search(u) or
            INTENTS["PROCEED_PAY"].search(u) or INTENTS["DEV_START"].search(u) or INTENTS["REFUND_STATUS"].search(u) or
            INTENTS["ADMIN_DIR_ENTER"].search(u) or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
            # resetea cancelación si estaba activa
            if self.ctx.state in ("CANCEL_INICIO","CANCEL_CONFIRMAR"):
                self.cancel_order_num = None
            return super().handle(user)

        # ---- “Marcar devolución recibida #ID” en cualquier momento ----
        m_rcv = INTENTS["RETURN_MARK_RECEIVED"].search(u)
        if m_rcv:
            rid_txt = m_rcv.group("rid")
            if not rid_txt:
                return "Dime el número de devolución, ej. 'marcar devolución #1 recibida'."
            out = mark_return_received(int(rid_txt))
            # después de marcar, muestra estatus del pedido más reciente
            msg = out["msg"]
            try:
                orders = list_orders()
                if orders:
                    msg += "\n\n" + refund_status(orders[0]["order_number"])
            except Exception:
                pass
            return msg

        # ---- Entrada a cancelación desde cualquier estado ----
        if INTENTS["CANCEL_START"].search(u) and self.ctx.state not in ("CANCEL_INICIO","CANCEL_CONFIRMAR"):
            self.ctx.state = "CANCEL_INICIO"
            return self.prompt()

        # ---- Flujo CANCEL_* ----
        if self.ctx.state == "CANCEL_INICIO":
            m = INTENTS["ORDER_ID_OR_RECENT"].search(u)
            if not m:
                ods = list_orders()
                if not ods:
                    self.ctx.state = "INICIO"; return "No hay pedidos para cancelar."
                lines = ["Pedidos (más recientes primero):"]
                for i,o in enumerate(ods,1):
                    lines.append(f"{i}) {o['order_number']} — ${int(o['total_cents'])/100:.2f} — {o['status']}")
                lines.append("Dime el número de pedido o 'el más reciente'.")
                return "\n".join(lines)

            key = m.group(1)
            if "reciente" in key or "ultimo" in key or "último" in key:
                ods = list_orders()
                if not ods:
                    self.ctx.state = "INICIO"; return "No hay pedidos registrados."
                chosen = ods[0]
            else:
                found = [o for o in list_orders() if o["order_number"] == key]
                if not found:
                    return "No ubico ese pedido. Verifica el formato 123-1234567-1234567."
                chosen = found[0]

            # elegibilidad
            if not cancel_eligible(chosen):
                self.ctx.state = "INICIO"
                return f"No puedes cancelar ese pedido (ventana de {CANCEL_WINDOW_MIN} min o ya procesado)."
            self.cancel_order_num = chosen["order_number"]
            self.ctx.state = "CANCEL_CONFIRMAR"
            return self.prompt()

        if self.ctx.state == "CANCEL_CONFIRMAR":
            if INTENTS["CONFIRM"].search(u):
                out = cancel_order(self.cancel_order_num)
                self.ctx.state = "INICIO"
                self.cancel_order_num = None
                return out["msg"] + "\n\n" + self.prompt()
            if INTENTS["DENY"].search(u):
                self.ctx.state = "INICIO"; self.cancel_order_num = None
                return "Cancelación abortada.\n\n" + self.prompt()
            return "¿Confirmas cancelar el pedido? (sí/no)"

        # fuera del flujo de cancelación, delega al comportamiento previo
        return super().handle(user)

print("✅ PASO 12 listo — Cancelación de pedido y completar reembolso al recibir devolución (AmazonBotV8)")

✅ PASO 12 listo — Cancelación de pedido y completar reembolso al recibir devolución (AmazonBotV8)


In [15]:
# PASO UX — Agregar CTA de "¿Continuamos?" en la pantalla de revisión sin tocar tu código previo

class AmazonBotV8UX(AmazonBotV8):
    def _review_text(self) -> str:
        # Usa el texto base de revisión (carrito + dirección + método + opciones)
        base = super()._review_text()
        # Añade un CTA claro para no tener que escribir "pagar" de nuevo
        cta = (
            "\n\n¿Continuamos con la operación? "
            "Escribe *'confirmar'* o *'realiza tu pedido y paga'*. "
            "También puedes decir *'cambiar dirección'* o *'cambiar método'*."
        )
        return base + cta

print("✅ Parche UX listo — Usa AmazonBotV8UX() para que la revisión pregunte si continúas")

✅ Parche UX listo — Usa AmazonBotV8UX() para que la revisión pregunte si continúas


In [20]:
# PASO 13 — "Ver mis pedidos" (listado + detalle) desde el inicio
import re
from datetime import datetime

# Intents para pedidos
INTENTS.update({
    "ORDERS_ENTER":   re.compile(r"\b(mis pedidos|ver pedidos|historial de pedidos|historial|pedidos)\b", re.I),
    "ORDER_BY_ID":    re.compile(rf"{ORDER_ID_RE}", re.I),  # ya teníamos ORDER_ID_RE antes
})

def _fmt_money(cents: int) -> str:
    return f"${cents/100:.2f}"

def _fmt_date(iso: str) -> str:
    try:
        return datetime.fromisoformat(iso).strftime("%Y-%m-%d %H:%M")
    except Exception:
        return iso or ""

def orders_list_text() -> str:
    ods = list_orders()
    if not ods:
        return "Aún no tienes pedidos."
    lines = ["Tus pedidos (más recientes primero):"]
    for i, o in enumerate(ods, 1):
        lines.append(f"{i}) {o['order_number']} — {_fmt_money(int(o['total_cents']))} — {o['status']} — {_fmt_date(o['created_at'])}")
    lines.append("\nResponde con el *número de la lista* (ej. '1') o pega el *número de pedido* (123-1234567-1234567).")
    lines.append("Comandos: 'atrás', 'salir'.")
    return "\n".join(lines)

def order_detail_text(order_number: str) -> str:
    ods = [o for o in list_orders() if o["order_number"] == order_number]
    if not ods:
        return "No encontré ese pedido."
    o = ods[0]
    items = list_order_items(order_number)
    lines = [f"*Pedido {order_number}*",
             f"Fecha: {_fmt_date(o['created_at'])}",
             f"Estatus: {o['status']}",
             f"Subtotal: {_fmt_money(int(o['subtotal_cents']))}",
             f"Envío: {_fmt_money(int(o['shipping_cents']))}",
             f"Total: {_fmt_money(int(o['total_cents']))}",
             "\nArtículos:"]
    for it in items:
        size = f" talla {it['size'].upper()}" if it.get("size") else ""
        line_total = it["unit_price_cents"] * it["qty"]
        lines.append(f"- {it['product_name']}{size} × {it['qty']} — {_fmt_money(it['unit_price_cents'])} c/u = {_fmt_money(line_total)}")
    lines.append("\nAcciones rápidas aquí mismo: escribe *'reembolso'* para ver el estatus de reembolso de este pedido, "
                 "o *'devolver'* para iniciar devolución de uno de sus artículos.")
    lines.append("También: 'atrás' (volver a la lista), 'salir'.")
    return "\n".join(lines)

class AmazonBotV9(AmazonBotV8UX):
    def __init__(self):
        super().__init__()
        self.orders_current = None  # número de pedido seleccionado

    def prompt(self) -> str:
        s = self.ctx.state
        if s == "ORDERS_LIST":
            return orders_list_text()
        if s == "ORDERS_DETAIL":
            return order_detail_text(self.orders_current) if self.orders_current else "No hay pedido seleccionado."
        return super().prompt()

    def _enter_orders(self) -> str:
        self.ctx.state = "ORDERS_LIST"
        self.orders_current = None
        return self.prompt()

    def _enter_order_detail(self, order_number: str) -> str:
        self.ctx.state = "ORDERS_DETAIL"
        self.orders_current = order_number
        return self.prompt()

    def handle(self, user: str) -> str:
        u = normalize(user)

        # --- Comandos globales que siempre respetamos ---
        if INTENTS["EXIT"].search(u):
            self.ctx.state = "FIN"; return "¡Gracias! Hasta luego."
        if INTENTS["ABUSIVE"].search(u):
            self.abusive_count += 1
            if self.abusive_count >= 2:
                self.ctx.state = "FIN"; return "Voy a terminar esta conversación por lenguaje inapropiado. Hasta luego."
            return "Mantengamos el respeto, por favor."
        if INTENTS["OFF_TOPIC"].search(u):
            # si está en flujo de pedidos, no salimos, solo avisamos
            if self.ctx.state in ("ORDERS_LIST", "ORDERS_DETAIL"):
                return "Ese tema no es de pedidos. Elige un pedido o di 'atrás' para salir."
            # fuera de pedidos → el manejo habitual
            return super().handle(user)

        # --- Entrada a "mis pedidos" desde cualquier estado ---
        if INTENTS["ORDERS_ENTER"].search(u) and self.ctx.state not in ("ORDERS_LIST","ORDERS_DETAIL"):
            return self._enter_orders()

        # --- Flujo de pedidos ---
        if self.ctx.state == "ORDERS_LIST":
            # atrás para salir a inicio
            if INTENTS["BACK"].search(u):
                self.ctx.state = "INICIO"; return self.prompt()
            # seleccionar por índice
            m = INTENTS["SELECT_N"].search(u)
            if m:
                idx = int(m.group("idx"))
                ods = list_orders()
                if not (1 <= idx <= len(ods)):
                    return "Índice inválido. Elige un número de la lista."
                return self._enter_order_detail(ods[idx-1]["order_number"])
            # seleccionar pegando número de pedido
            m = INTENTS["ORDER_BY_ID"].search(u) or INTENTS["ORDER_ID_OR_RECENT"].search(u)
            if m:
                key = m.group(0) if "ORDER_BY_ID" in INTENTS else m.group(1)
                if "reciente" in key or "ultimo" in key or "último" in key:
                    ods = list_orders()
                    if not ods: 
                        self.ctx.state = "INICIO"; return "No hay pedidos registrados."
                    return self._enter_order_detail(ods[0]["order_number"])
                # búsqueda exacta
                ods = [o for o in list_orders() if o["order_number"] == key]
                if not ods:
                    return "No ubico ese pedido. Verifica el formato 123-1234567-1234567."
                return self._enter_order_detail(ods[0]["order_number"])
            # otros comandos (seguir comprando, ver carrito, etc.) — salimos de pedidos y delegamos
            if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or INTENTS["PAY"].search(u)
                or INTENTS["PROCEED_PAY"].search(u) or INTENTS["DEV_START"].search(u)
                or INTENTS["REFUND_STATUS"].search(u) or INTENTS["ADMIN_DIR_ENTER"].search(u)
                or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
                self.ctx.state = "INICIO"
                return super().handle(user)
            return self.prompt()

        if self.ctx.state == "ORDERS_DETAIL":
            # volver a la lista
            if INTENTS["BACK"].search(u):
                return self._enter_orders()
            # atajo: reembolso de este pedido
            if INTENTS["REFUND_STATUS"].search(u):
                msg = refund_status(self.orders_current)
                return msg + "\n\n" + self.prompt()
            # atajo: iniciar devolución de este pedido
            if INTENTS["DEV_START"].search(u):
                # comprobar elegibilidad de 30 días (como en devoluciones)
                row = next((o for o in list_orders() if o["order_number"] == self.orders_current), None)
                if not row:
                    return "No ubico el pedido en tu historial."
                if not return_eligible(row):
                    # salir a INICIO con mensaje
                    self.ctx.state = "INICIO"
                    return "Ese pedido ya no es elegible para devolución (ventana de 30 días)." + "\n\n" + self.prompt()
                # inicializamos datos de devolución y saltamos directo a la selección de artículo
                self.dev = DevData(
                    order=self.orders_current,
                    items=list_order_items(self.orders_current),
                    item_idx=None, qty=None, reason=None, method=None
                )
                self.ctx.state = "DEV_LISTA_ITEMS"
                return super().prompt()
            # otros comandos (compras/checkout/admin) — salir de pedidos y delegar
            if (INTENTS["VIEW_CART"].search(u) or INTENTS["SEARCH"].search(u) or INTENTS["PAY"].search(u)
                or INTENTS["PROCEED_PAY"].search(u) or INTENTS["ADMIN_DIR_ENTER"].search(u)
                or INTENTS["ADMIN_PAGO_ENTER"].search(u)):
                self.ctx.state = "INICIO"
                return super().handle(user)
            # si escribe el número de otro pedido, mostrar ese
            m = INTENTS["ORDER_BY_ID"].search(u)
            if m:
                return self._enter_order_detail(m.group(0))
            # repetir detalle por defecto
            return self.prompt()

        # --- fuera del flujo de pedidos, comportamiento normal ---
        return super().handle(user)

print("✅ PASO 13 listo — 'Mis pedidos' integrado (AmazonBotV9)")

✅ PASO 13 listo — 'Mis pedidos' integrado (AmazonBotV9)


In [ ]:
bot = AmazonBotV9()
print("👋 Hola, ahora puedes ver tus pedidos desde el inicio: di 'mis pedidos'.\n")
print(bot.prompt())

while True:
    try:
        user = input("\nTú: ")
    except EOFError:
        break
    resp = bot.handle(user)
    print("\nBot:", resp)
    if bot.ctx.state == "FIN":
        break

👋 Hola, ahora puedes ver tus pedidos desde el inicio: di 'mis pedidos'.

¿Qué estás buscando? (ej. 'playera', 'sudadera', 'cafetera')
También: 'ver carrito', 'pagar', 'ayuda', 'salir'.



Tú:  holii



Bot: Resultados:
1) Playera básica unisex — $199.00 (tallas)
2) Sudadera con gorro — $599.00 (tallas)
3) Cafetera de goteo 12 tazas — $1499.00

Elige un producto por número (ej. '1'), o escribe otra búsqueda.



Tú:  no



Bot: Resultados:
1) Playera básica unisex — $199.00 (tallas)
2) Sudadera con gorro — $599.00 (tallas)
3) Cafetera de goteo 12 tazas — $1499.00

Elige un producto por número (ej. '1'), o escribe otra búsqueda.


...


Bot: Resultados:
1) Playera básica unisex — $199.00 (tallas)
2) Sudadera con gorro — $599.00 (tallas)
3) Cafetera de goteo 12 tazas — $1499.00

Elige un producto por número (ej. '1'), o escribe otra búsqueda.



Tú:  atras



Bot: ¿Qué estás buscando? (ej. 'playera', 'sudadera', 'cafetera')
También: 'ver carrito', 'pagar', 'ayuda', 'salir'.



Tú:  atras



Bot: ¿Qué estás buscando? (ej. 'playera', 'sudadera', 'cafetera')
También: 'ver carrito', 'pagar', 'ayuda', 'salir'.
